# Data Fusion 2026 — hybrid stack notebook (exact uploaded NN on GPU, XGB/CAT/PyBoost-meta on GPU, LGBM/PyBoost checkpoints)

In [1]:

# ============================================================
# 0) CONFIG / INSTALL / PATHS
# ============================================================
import os
import sys
import gc
import json
import math
import time
import copy
import random
import shutil
import hashlib
import warnings
import subprocess
from pathlib import Path

warnings.filterwarnings("ignore")

DATA_DIR = Path("/kaggle/input/data-fusion-contest-2026")
CKPT_DIR = Path("/kaggle/input/datasets/dderggft/checks/checks/checks")

WORK_DIR = Path("/kaggle/working/df2026_hybrid_stack_bestnn_exact_gpu_pyboostmeta")
FEATURES_DIR = WORK_DIR / "features"
CACHE_DIR = WORK_DIR / "cache"
SUB_DIR = WORK_DIR / "submissions"
NN_CKPT_DIR = CACHE_DIR / "checkpoints_nn_best"
BASE_PRED_CKPT_DIR = CACHE_DIR / "base_pred_checkpoints"

for p in [WORK_DIR, FEATURES_DIR, CACHE_DIR, SUB_DIR, NN_CKPT_DIR, BASE_PRED_CKPT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)

FORCE_REBUILD_FEATURES = False

N_FOLDS_BASE = 5
N_FOLDS_META = 5
N_FOLDS_FINAL = 5

USE_LGBM_CHECKPOINT = True
USE_PYBOOST_CHECKPOINT = True
LGBM_CKPT_PATH = CKPT_DIR / "lgbm_predictions (2).npz"
PYBOOST_CKPT_PATH = CKPT_DIR / "pyboost_predictions.npz"

# ---------- internal checkpoints for retrained base models ----------
NN_BASE_PRED_CKPT_PATH = CKPT_DIR / "nn_best_predictions.npz"
XGB_BASE_PRED_CKPT_PATH = CKPT_DIR / "xgboost_predictions.npz"
CAT_BASE_PRED_CKPT_PATH = Path("/kaggle/input/datasets/dderggft/checks/catboost_predictions.npz")
EXTERNAL_CACHE_DIRS = [
    CKPT_DIR,
    CKPT_DIR.parent,
    CKPT_DIR.parent.parent,
    Path("/kaggle/input/datasets/dderggft/checks"),
]
ENABLE_NN = False
ENABLE_XGB = False
ENABLE_CATBOOST = False
ENABLE_LGBM = False
ENABLE_PYBOOST = False   # base PyBoost переобучать не надо: берём checkpoint

USE_CACHE = True
FORCE_REBUILD_BASE = False
FORCE_REBUILD_META = False
FORCE_REBUILD_FINAL = False

USE_NN_CHECKPOINT = True
USE_XGB_CHECKPOINT = True
USE_CATBOOST_CHECKPOINT = True
# ---------- device policy ----------
FORCE_NN_CPU = False
FORCE_LGB_CPU = True
FORCE_XGB_GPU = True
FORCE_CAT_GPU = True
FORCE_PYBOOST_META_GPU = True

# ---------- optional pyboost meta install ----------
ENABLE_PYBOOST_META = True
INSTALL_PYBOOST_META = True
CUPY_PACKAGE = "cupy-cuda12x"

# ---------- base model params ----------
XGB_ESTIMATORS = 3000
XGB_LR = 0.03
XGB_EARLY_STOP = 120
XGB_MAX_BIN = 256

CAT_ITERATIONS = 3500
CAT_LR = 0.03
CAT_EARLY_STOP = 120

# ---------- NN (best uploaded variant) ----------
NN_BATCH_SIZE = 1024
NN_EPOCHS = 50
NN_LR = 5e-4
NN_WEIGHT_DECAY = 1e-3
NN_PATIENCE = 10
NN_GRAD_CLIP = 1.0
NN_HIDDEN_DIM = 384

DAE_BOTTLENECK_DIM = 256
DAE_EPOCHS = 20
DAE_LR = 1e-3
DAE_BATCH_SIZE = 4096
DAE_CORRUPTION_RATE = 0.15
DAE_WEIGHT_DECAY = 1e-5
DAE_GRAD_CLIP = 1.0

K_ENSEMBLE = 8
ASL_GAMMA_NEG = 4
ASL_GAMMA_POS = 1
ASL_CLIP = 0.05
MIXUP_ALPHA = 0.2
PLR_N_BINS = 24
TOP_K_SWA = 5

# ---------- meta learners ----------
ENABLE_L1 = True
ENABLE_L2 = True
ENABLE_L3 = True
ENABLE_LGB_META = False
ENABLE_XGB_META = True
ENABLE_CAT_META = True

PYBOOST_META_NTREES = 2500
PYBOOST_META_LR = 0.03
PYBOOST_META_VERBOSE = 200
PYBOOST_META_EARLY_STOP = 120
PYBOOST_META_MAX_DEPTH = 6
PYBOOST_META_MAX_BIN = 256

MAX_L1_MODELS_PER_TARGET = 5
MAX_L2_MODELS_PER_TARGET = 5
MAX_L3_MODELS_PER_TARGET = 5
MAX_FINAL_MODELS_PER_TARGET = 6

COARSE_GRID = [round(x, 2) for x in list(__import__("numpy").arange(0.00, 1.0001, 0.05))]
FINE_GRID = [round(x, 2) for x in list(__import__("numpy").arange(0.00, 1.0001, 0.02))]
HOLDOUT_GRID = [round(x, 2) for x in list(__import__("numpy").arange(0.02, 0.9801, 0.02))]

def ensure_packages(packages):
    import importlib
    missing = []
    for pkg in packages:
        mod_name = pkg.split("==")[0].split(">=")[0]
        import_name = {"scikit-learn": "sklearn", "pyarrow": "pyarrow", "py-boost": "py_boost"}.get(mod_name, mod_name)
        try:
            importlib.import_module(import_name)
        except Exception:
            missing.append(pkg)

    if missing:
        print("Installing missing packages:", missing)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)

base_packages = [
    "numpy", "pandas", "polars", "pyarrow", "scipy", "scikit-learn",
    "lightgbm", "xgboost", "catboost", "torch",
]
ensure_packages(base_packages)

if ENABLE_PYBOOST_META and INSTALL_PYBOOST_META:
    ensure_packages([CUPY_PACKAGE, "py-boost"])

print("DATA_DIR    :", DATA_DIR)
print("CKPT_DIR    :", CKPT_DIR)
print("WORK_DIR    :", WORK_DIR)
print("FEATURES_DIR:", FEATURES_DIR)
print("CACHE_DIR   :", CACHE_DIR)
print("SUB_DIR     :", SUB_DIR)
print("NN_CKPT_DIR :", NN_CKPT_DIR)


Installing missing packages: ['cupy-cuda12x', 'py-boost']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 129.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.7/198.7 kB 16.1 MB/s eta 0:00:00
DATA_DIR    : /kaggle/input/data-fusion-contest-2026
CKPT_DIR    : /kaggle/input/datasets/dderggft/checks/checks/checks
WORK_DIR    : /kaggle/working/df2026_hybrid_stack_bestnn_exact_gpu_pyboostmeta
FEATURES_DIR: /kaggle/working/df2026_hybrid_stack_bestnn_exact_gpu_pyboostmeta/features
CACHE_DIR   : /kaggle/working/df2026_hybrid_stack_bestnn_exact_gpu_pyboostmeta/cache
SUB_DIR     : /kaggle/working/df2026_hybrid_stack_bestnn_exact_gpu_pyboostmeta/submissions
NN_CKPT_DIR : /kaggle/working/df2026_hybrid_stack_bestnn_exact_gpu_pyboostmeta/cache/checkpoints_nn_best


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuml-cu12 25.6.0 requires treelite==4.4.1, but you have treelite 3.9.1 which is incompatible.


In [2]:
# ============================================================
# 1) 01_feature_engineering (adapted to notebook)
# ============================================================
import numpy as np
import pandas as pd
import polars as pl

from scipy.sparse import csr_matrix, hstack as sparse_hstack
from sklearn.decomposition import TruncatedSVD

NULL_PCA_COMPONENTS = 20
INDIVIDUAL_NULL_THRESHOLD = 0.05

NULL_GROUPS_MAIN = {
    "null_group_A": "num_feature_1",
    "null_group_B": "num_feature_8",
    "null_group_C": "num_feature_11",
    "null_group_D": "num_feature_38",
    "null_group_E": "num_feature_72",
}

NULL_GROUPS_EXTRA = {
    "null_extra_E4": "num_feature_155",
    "null_extra_E5": "num_feature_140",
    "null_extra_E8": "num_feature_139",
    "null_extra_E9": "num_feature_222",
    "null_extra_E10": "num_feature_134",
    "null_extra_E11": "num_feature_158",
    "null_extra_E12": "num_feature_240",
    "null_extra_E13": "num_feature_194",
    "null_extra_E14": "num_feature_197",
    "null_extra_E15": "num_feature_167",
}

NUM_DIFFS = [
    ("num_feature_7", "num_feature_42"),
    ("num_feature_33", "num_feature_42"),
    ("num_feature_36", "num_feature_42"),
    ("num_feature_7", "num_feature_132"),
    ("num_feature_33", "num_feature_29"),
    ("num_feature_7", "num_feature_57"),
    ("num_feature_33", "num_feature_132"),
    ("num_feature_7", "num_feature_125"),
    ("num_feature_29", "num_feature_42"),
    ("num_feature_41", "num_feature_42"),
    ("num_feature_56", "num_feature_34"),
    ("num_feature_7", "num_feature_34"),
    ("num_feature_33", "num_feature_13"),
    ("num_feature_7", "num_feature_118"),
    ("num_feature_36", "num_feature_34"),
    ("num_feature_29", "num_feature_13"),
    ("num_feature_118", "num_feature_42"),
    ("num_feature_125", "num_feature_118"),
]

CAT_INTERACTIONS = [
    ("cat_feature_66", "cat_feature_46"),
    ("cat_feature_66", "cat_feature_39"),
    ("cat_feature_66", "cat_feature_48"),
    ("cat_feature_66", "cat_feature_9"),
    ("cat_feature_66", "cat_feature_52"),
]

RATIO_FEATURES = [
    ("num_feature_62", "num_feature_79"),
    ("num_feature_27", "num_feature_79"),
    ("num_feature_76", "num_feature_79"),
    ("num_feature_62", "num_feature_86"),
    ("num_feature_116", "num_feature_124"),
    ("num_feature_36", "num_feature_79"),
    ("num_feature_41", "num_feature_79"),
    ("num_feature_83", "num_feature_108"),
    ("num_feature_83", "num_feature_103"),
]

DUPLICATE_CATS = [
    "cat_feature_24",
    "cat_feature_25",
    "cat_feature_26",
    "cat_feature_29",
    "cat_feature_50",
    "cat_feature_63",
]

def filter_extra_features(train_extra, test_extra):
    extra_cols = [c for c in train_extra.columns if c != "customer_id"]
    n_rows = train_extra.height
    to_drop = set()
    for col in extra_cols:
        s = train_extra[col]
        if s.null_count() / n_rows > 0.99:
            to_drop.add(col)
            continue
        non_null = s.drop_nulls()
        if len(non_null) == 0 or non_null.n_unique() <= 1:
            to_drop.add(col)
    keep_cols = ["customer_id"] + [c for c in extra_cols if c not in to_drop]
    print(f"  Extra: {len(extra_cols)} total, {len(to_drop)} removed, {len(keep_cols)-1} kept")
    return train_extra.select(keep_cols), test_extra.select(keep_cols)

def deduplicate_extra_features(train_extra, test_extra):
    extra_cols = [c for c in train_extra.columns if c.startswith("num_feature")]
    seen_hashes = {}
    dup_cols = []
    BATCH = 200
    for start in range(0, len(extra_cols), BATCH):
        batch_cols = extra_cols[start:start + BATCH]
        batch_np = train_extra.select(batch_cols).to_numpy()
        for i, col in enumerate(batch_cols):
            h = hashlib.md5(batch_np[:, i].tobytes()).hexdigest()
            if h in seen_hashes:
                dup_cols.append(col)
            else:
                seen_hashes[h] = col
        del batch_np
    if dup_cols:
        train_extra = train_extra.drop(dup_cols)
        test_extra = test_extra.drop(dup_cols)
        print(f"  Dedup: removed {len(dup_cols)} -> {len(extra_cols)-len(dup_cols)} unique")
    return train_extra, test_extra

def null_pattern_pca(train_extra_raw, test_extra_raw, n_components=20):
    all_cols = [c for c in train_extra_raw.columns if c.startswith("num_feature")]
    n_train = train_extra_raw.height
    BATCH = 500
    sparse_blocks = []
    for start in range(0, len(all_cols), BATCH):
        batch_cols = all_cols[start:start + BATCH]
        train_np = train_extra_raw.select(batch_cols).to_numpy()
        test_np = test_extra_raw.select(batch_cols).to_numpy()
        combined = np.vstack([train_np, test_np])
        del train_np, test_np
        null_bits = np.isnan(combined).astype(np.float32)
        sparse_blocks.append(csr_matrix(null_bits))
        del combined, null_bits
    null_sparse = sparse_hstack(sparse_blocks, format="csr")
    del sparse_blocks
    gc.collect()

    svd = TruncatedSVD(n_components=n_components, random_state=SEED)
    pca_features = svd.fit_transform(null_sparse)
    var_explained = svd.explained_variance_ratio_.sum()
    print(f"  Null PCA: {null_sparse.shape[1]} cols -> {n_components} components, var explained: {var_explained:.3f}")

    del null_sparse
    gc.collect()

    return pca_features[:n_train].astype(np.float32), pca_features[n_train:].astype(np.float32)

def add_null_indicators(df, groups_dict):
    new_cols = []
    for name, ref_col in groups_dict.items():
        if ref_col in df.columns:
            new_cols.append(pl.col(ref_col).is_null().cast(pl.Int8).alias(name))
    if new_cols:
        df = df.with_columns(new_cols)
    return df

def select_null_indicator_features(train_main, num_cols, train_tgt, target_cols, threshold):
    null_arrays, valid_cols = [], []
    for col in num_cols:
        arr = train_main[col].is_null().cast(pl.Float32).to_numpy()
        if arr.std() < 1e-8:
            continue
        null_arrays.append(arr)
        valid_cols.append(col)
    if not null_arrays:
        return []

    null_matrix = np.column_stack(null_arrays).astype(np.float32)
    tgt_np = train_tgt.select(target_cols).to_numpy().astype(np.float32)

    null_mean = null_matrix.mean(axis=0, keepdims=True)
    null_std = null_matrix.std(axis=0, keepdims=True)
    null_std[null_std < 1e-8] = 1.0
    null_matrix = (null_matrix - null_mean) / null_std

    tgt_mean = tgt_np.mean(axis=0, keepdims=True)
    tgt_std = tgt_np.std(axis=0, keepdims=True)
    tgt_std[tgt_std < 1e-8] = 1.0
    tgt_np = (tgt_np - tgt_mean) / tgt_std

    corr_matrix = (null_matrix.T @ tgt_np) / null_matrix.shape[0]
    max_abs_corr = np.abs(corr_matrix).max(axis=1)
    return [col for col, c in zip(valid_cols, max_abs_corr) if c > threshold]

def add_individual_null_indicators(df, selected_cols):
    exprs = [pl.col(c).is_null().cast(pl.Int8).alias(f"is_null_{c}") for c in selected_cols]
    return df.with_columns(exprs)

def add_frequency_encoding(train_df, test_df, cat_cols):
    total = train_df.height + test_df.height
    freq_cols = []
    for col in cat_cols:
        fname = f"freq_{col}"
        combined = pl.concat([train_df.select(col), test_df.select(col)])
        freq_map = (
            combined[col]
            .value_counts()
            .with_columns((pl.col("count") / total).alias(fname))
            .select([col, fname])
        )
        train_df = train_df.join(freq_map, on=col, how="left")
        test_df = test_df.join(freq_map, on=col, how="left")
        freq_cols.append(fname)
    return train_df, test_df, freq_cols

def add_cat_interaction_freqs(train_df, test_df, interactions):
    total = train_df.height + test_df.height
    freq_cols = []
    for c1, c2 in interactions:
        fname = f"freq_{c1}_x_{c2}"
        combo = pl.col(c1).cast(pl.Int64) * 1_000_000 + pl.col(c2).cast(pl.Int64)
        combined = pl.concat([
            train_df.select(combo.alias("_combo")),
            test_df.select(combo.alias("_combo")),
        ])
        freq_map = (
            combined["_combo"]
            .value_counts()
            .with_columns((pl.col("count") / total).alias(fname))
            .select(["_combo", fname])
        )
        train_df = train_df.with_columns(combo.alias("_combo")).join(freq_map, on="_combo", how="left").drop("_combo")
        test_df = test_df.with_columns(combo.alias("_combo")).join(freq_map, on="_combo", how="left").drop("_combo")
        freq_cols.append(fname)
    return train_df, test_df, freq_cols

def add_numerical_diffs(df, diff_pairs):
    exprs = []
    for a, b in diff_pairs:
        if a in df.columns and b in df.columns:
            a_id = a.replace("num_feature_", "")
            b_id = b.replace("num_feature_", "")
            exprs.append((pl.col(a) - pl.col(b)).alias(f"diff_{a_id}_minus_{b_id}"))
    if exprs:
        df = df.with_columns(exprs)
    return df

def add_ratio_features(df, ratio_pairs):
    exprs = []
    for a, b in ratio_pairs:
        if a in df.columns and b in df.columns:
            name = f"ratio_{a.replace('num_feature_', '')}_{b.replace('num_feature_', '')}"
            exprs.append(
                pl.when(pl.col(b).abs() > 1e-8)
                .then(pl.col(a) / pl.col(b))
                .otherwise(None)
                .alias(name)
            )
    if exprs:
        df = df.with_columns(exprs)
    return df

def add_null_count(df, cols, name, batch_size=300):
    if len(cols) == 0:
        return df.with_columns(pl.lit(0).cast(pl.UInt16).alias(name))
    parts = []
    for i in range(0, len(cols), batch_size):
        batch = cols[i:i + batch_size]
        parts.append(pl.sum_horizontal([pl.col(c).is_null().cast(pl.UInt16) for c in batch]))
    total = parts[0]
    for p in parts[1:]:
        total = total + p
    return df.with_columns(total.alias(name))

def add_row_mean(df, num_cols):
    return df.with_columns(pl.mean_horizontal([pl.col(c) for c in num_cols]).alias("row_mean_main"))

def add_row_stats(df, num_cols, prefix):
    arr = df.select(num_cols).to_numpy()
    row_mean = np.nanmean(arr, axis=1, keepdims=True)
    diff = arr - row_mean
    m2 = np.nanmean(diff ** 2, axis=1)
    m3 = np.nanmean(diff ** 3, axis=1)
    row_std = np.sqrt(np.maximum(m2, 0)).astype(np.float32)
    with np.errstate(divide="ignore", invalid="ignore"):
        row_skew = np.where(m2 > 1e-16, m3 / (m2 ** 1.5 + 1e-16), 0).astype(np.float32)
    del arr, diff, m2, m3
    gc.collect()
    return df.with_columns([
        pl.Series(f"{prefix}_row_std", row_std),
        pl.Series(f"{prefix}_row_skew", row_skew),
    ])

def remove_duplicate_cats(train_df, test_df, dup_cols):
    existing = [c for c in dup_cols if c in train_df.columns]
    if existing:
        train_df = train_df.drop(existing)
        test_df = test_df.drop(existing)
        print(f"  Removed {len(existing)} duplicate cats")
    return train_df, test_df

def run_feature_engineering(force=False):
    train_fp = FEATURES_DIR / "train_features.parquet"
    test_fp = FEATURES_DIR / "test_features.parquet"
    meta_fp = FEATURES_DIR / "meta.json"
    targets_fp = FEATURES_DIR / "targets.parquet"

    if (not force) and train_fp.exists() and test_fp.exists() and meta_fp.exists() and targets_fp.exists():
        print("Feature engineering already exists -> skip rebuild")
        return

    t0 = time.time()
    print("=" * 60)
    print("Step 1: Feature Engineering")
    print("=" * 60)

    print("\n[1/8] Loading raw data...")
    train_main = pl.read_parquet(DATA_DIR / "train_main_features.parquet")
    test_main = pl.read_parquet(DATA_DIR / "test_main_features.parquet")
    train_extra_raw = pl.read_parquet(DATA_DIR / "train_extra_features.parquet")
    test_extra_raw = pl.read_parquet(DATA_DIR / "test_extra_features.parquet")
    train_tgt = pl.read_parquet(DATA_DIR / "train_target.parquet")

    cat_cols = sorted([c for c in train_main.columns if c.startswith("cat_feature")])
    num_cols_main = sorted([c for c in train_main.columns if c.startswith("num_feature")])
    target_cols = [c for c in train_tgt.columns if c.startswith("target_")]

    print("\n[2/8] Null Pattern PCA...")
    train_null_pca, test_null_pca = null_pattern_pca(train_extra_raw, test_extra_raw, NULL_PCA_COMPONENTS)

    print("\n[3/8] Filter + dedup extra features...")
    train_extra, test_extra = filter_extra_features(train_extra_raw, test_extra_raw)
    del train_extra_raw, test_extra_raw
    gc.collect()
    train_extra, test_extra = deduplicate_extra_features(train_extra, test_extra)

    print("\n[4/8] Prepare categoricals...")
    train_main, test_main = remove_duplicate_cats(train_main, test_main, DUPLICATE_CATS)
    cat_cols = sorted([c for c in train_main.columns if c.startswith("cat_feature")])
    train_main = train_main.with_columns(pl.col(cat_cols).cast(pl.Int32))
    test_main = test_main.with_columns(pl.col(cat_cols).cast(pl.Int32))

    print("\n[5/8] Engineering features...")

    train_main = add_null_indicators(train_main, NULL_GROUPS_MAIN)
    test_main = add_null_indicators(test_main, NULL_GROUPS_MAIN)
    train_extra = add_null_indicators(train_extra, NULL_GROUPS_EXTRA)
    test_extra = add_null_indicators(test_extra, NULL_GROUPS_EXTRA)

    train_main = add_null_count(train_main, num_cols_main, "null_count_main")
    test_main = add_null_count(test_main, num_cols_main, "null_count_main")
    extra_num_cols = [c for c in train_extra.columns if c.startswith("num_feature")]
    train_extra = add_null_count(train_extra, extra_num_cols, "null_count_extra")
    test_extra = add_null_count(test_extra, extra_num_cols, "null_count_extra")

    train_main, test_main, freq_cols = add_frequency_encoding(train_main, test_main, cat_cols)
    train_main, test_main, interact_cols = add_cat_interaction_freqs(train_main, test_main, CAT_INTERACTIONS)

    train_main = add_numerical_diffs(train_main, NUM_DIFFS)
    test_main = add_numerical_diffs(test_main, NUM_DIFFS)

    train_main = add_ratio_features(train_main, RATIO_FEATURES)
    test_main = add_ratio_features(test_main, RATIO_FEATURES)

    train_main = add_row_mean(train_main, num_cols_main)
    test_main = add_row_mean(test_main, num_cols_main)

    null_ind_cols = select_null_indicator_features(
        train_main, num_cols_main, train_tgt, target_cols, INDIVIDUAL_NULL_THRESHOLD
    )
    train_main = add_individual_null_indicators(train_main, null_ind_cols)
    test_main = add_individual_null_indicators(test_main, null_ind_cols)

    train_main = add_row_stats(train_main, num_cols_main, "main")
    test_main = add_row_stats(test_main, num_cols_main, "main")

    extra_num_for_stats = [c for c in train_extra.columns if c.startswith("num_feature")]
    train_extra = add_row_stats(train_extra, extra_num_for_stats, "extra")
    test_extra = add_row_stats(test_extra, extra_num_for_stats, "extra")

    print("\n[6/8] Joining features...")
    train_feat = train_main.join(train_extra, on="customer_id")
    test_feat = test_main.join(test_extra, on="customer_id")
    del train_main, train_extra, test_main, test_extra
    gc.collect()

    null_pca_cols = [f"null_pca_{i}" for i in range(NULL_PCA_COMPONENTS)]
    train_feat = train_feat.hstack(pl.DataFrame(train_null_pca, schema=null_pca_cols))
    test_feat = test_feat.hstack(pl.DataFrame(test_null_pca, schema=null_pca_cols))
    del train_null_pca, test_null_pca

    feature_cols = [c for c in train_feat.columns if c != "customer_id"]
    cat_feature_names = [c for c in feature_cols if c.startswith("cat_feature")]
    num_feature_names = [c for c in feature_cols if c not in cat_feature_names]

    print(f"  Total: {len(feature_cols)} features ({len(cat_feature_names)} cat, {len(num_feature_names)} num)")

    print("\n[7/8] Saving targets...")
    targets = train_tgt.select(["customer_id"] + target_cols)
    targets.write_parquet(FEATURES_DIR / "targets.parquet")

    print("\n[8/8] Saving features...")
    train_feat.write_parquet(FEATURES_DIR / "train_features.parquet")
    test_feat.write_parquet(FEATURES_DIR / "test_features.parquet")

    meta = {
        "cat_cols": cat_feature_names,
        "num_cols": num_feature_names,
        "feature_names": feature_cols,
        "target_cols": target_cols,
    }
    with open(FEATURES_DIR / "meta.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    elapsed = time.time() - t0
    print(f"\nDone in {elapsed / 60:.1f} min.")
    print(f"Saved to: {FEATURES_DIR}")

In [3]:
# ============================================================
# 2) RUN FEATURE ENGINEERING + LOAD FEATURES (FIXED)
# ============================================================
from pathlib import Path
import os
import shutil
import json
import polars as pl

REQUIRED_RAW_FILES = [
    "train_main_features.parquet",
    "test_main_features.parquet",
    "train_extra_features.parquet",
    "test_extra_features.parquet",
    "train_target.parquet",
    "sample_submit.parquet",
]

def _candidate_search_roots():
    roots = []

    # текущий DATA_DIR, если уже задан
    if "DATA_DIR" in globals():
        try:
            roots.append(Path(DATA_DIR))
        except Exception:
            pass

    # типичные kaggle locations
    roots.extend([
        Path("/kaggle/input"),
        Path("/kaggle/input/data-fusion-contest-2026"),
        Path("/kaggle/input/data-fusion-2026"),
        Path("/kaggle/input/datafusion2026"),
        Path.cwd(),
        Path.cwd().parent,
    ])

    # убрать дубли
    uniq = []
    seen = set()
    for p in roots:
        p = Path(p)
        if str(p) not in seen:
            uniq.append(p)
            seen.add(str(p))
    return uniq

def _find_file_by_name(filename, search_roots):
    # 1) сначала прямое попадание
    for root in search_roots:
        p = root / filename
        if p.exists():
            return p

    # 2) затем рекурсивный поиск
    for root in search_roots:
        if not root.exists():
            continue
        try:
            matches = list(root.rglob(filename))
            if matches:
                # берем самый короткий путь как наиболее "верхний"/прямой
                matches = sorted(matches, key=lambda x: (len(x.parts), str(x)))
                return matches[0]
        except Exception:
            continue

    return None

def _resolve_and_stage_raw_files():
    search_roots = _candidate_search_roots()
    found = {}

    print("Searching raw parquet files...")
    for fname in REQUIRED_RAW_FILES:
        p = _find_file_by_name(fname, search_roots)
        if p is None:
            raise FileNotFoundError(
                f"Не найден файл: {fname}\n"
                f"Проверьте DATA_DIR или имя kaggle dataset.\n"
                f"Поиск выполнялся в: {[str(x) for x in search_roots]}"
            )
        found[fname] = p
        print(f"  {fname:30s} -> {p}")

    # staging dir, чтобы run_feature_engineering() мог читать стандартные имена
    raw_stage_dir = WORK_DIR / "raw_autoresolved"
    raw_stage_dir.mkdir(parents=True, exist_ok=True)

    for fname, src in found.items():
        dst = raw_stage_dir / fname
        if dst.exists():
            continue
        try:
            os.symlink(src, dst)
        except Exception:
            shutil.copy2(src, dst)

    return raw_stage_dir, found

# ---- 1) найти реальные raw parquet и подложить в единый DATA_DIR ----
RAW_STAGE_DIR, RAW_FILE_PATHS = _resolve_and_stage_raw_files()

# ВАЖНО: обновляем глобальный DATA_DIR, чтобы run_feature_engineering() использовал его
DATA_DIR = RAW_STAGE_DIR
print("\nResolved DATA_DIR:", DATA_DIR)

# ---- 2) запускаем FE ----
run_feature_engineering(force=FORCE_REBUILD_FEATURES)

# ---- 3) грузим уже готовые engineered features ----
train_features_path = FEATURES_DIR / "train_features.parquet"
test_features_path = FEATURES_DIR / "test_features.parquet"
targets_path = FEATURES_DIR / "targets.parquet"
meta_path = FEATURES_DIR / "meta.json"

for p in [train_features_path, test_features_path, targets_path, meta_path]:
    if not p.exists():
        raise FileNotFoundError(f"Ожидался файл после feature engineering, но его нет: {p}")

train_feat_pl = pl.read_parquet(train_features_path)
test_feat_pl = pl.read_parquet(test_features_path)
targets_pl = pl.read_parquet(targets_path)

# sample_submit читаем из реально найденного raw-файла
sample_submit = pl.read_parquet(RAW_FILE_PATHS["sample_submit.parquet"])

with open(meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

cat_cols = meta["cat_cols"]
num_cols = meta["num_cols"]
feature_cols = meta["feature_names"]
target_cols = meta["target_cols"]
predict_cols = [c.replace("target_", "predict_") for c in target_cols]

print("\nLoaded engineered artifacts:")
print("train_feat:", train_feat_pl.shape)
print("test_feat :", test_feat_pl.shape)
print("targets   :", targets_pl.shape)
print("cat_cols  :", len(cat_cols))
print("num_cols  :", len(num_cols))
print("targets   :", len(target_cols))
print("sample    :", sample_submit.shape)

Searching raw parquet files...
  train_main_features.parquet    -> /kaggle/input/datasets/hatab123/data-fusion-contest-2026/train_main_features.parquet
  test_main_features.parquet     -> /kaggle/input/datasets/hatab123/data-fusion-contest-2026/test_main_features.parquet
  train_extra_features.parquet   -> /kaggle/input/datasets/hatab123/data-fusion-contest-2026/train_extra_features.parquet
  test_extra_features.parquet    -> /kaggle/input/datasets/hatab123/data-fusion-contest-2026/test_extra_features.parquet
  train_target.parquet           -> /kaggle/input/datasets/hatab123/data-fusion-contest-2026/train_target.parquet
  sample_submit.parquet          -> /kaggle/input/datasets/hatab123/data-fusion-contest-2026/sample_submit.parquet

Resolved DATA_DIR: /kaggle/working/df2026_hybrid_stack_bestnn_exact_gpu_pyboostmeta/raw_autoresolved
Step 1: Feature Engineering

[1/8] Loading raw data...

[2/8] Null Pattern PCA...
  Null PCA: 2241 cols -> 20 components, var explained: 0.907

[3/8] Filt

In [4]:
# ============================================================
# 3) HELPERS / METRICS / MATRIX PREP
# ============================================================
from itertools import combinations
from concurrent.futures import ThreadPoolExecutor, as_completed

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold

CPU_COUNT = max(1, os.cpu_count() or 1)
PARALLEL_TARGET_WORKERS = max(1, min(CPU_COUNT, 8))
CPU_META_TARGETS_PARALLEL = max(1, min(4, CPU_COUNT))
CPU_META_THREADS_PER_MODEL = max(1, CPU_COUNT // CPU_META_TARGETS_PARALLEL)

os.environ["OMP_NUM_THREADS"] = str(CPU_COUNT)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_COUNT)
os.environ["MKL_NUM_THREADS"] = str(CPU_COUNT)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_COUNT)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_COUNT)

def _get_cache_search_dirs():
    dirs = [CACHE_DIR]

    for p in EXTERNAL_CACHE_DIRS:
        try:
            dirs.append(Path(p))
        except Exception:
            pass

    uniq = []
    seen = set()
    for p in dirs:
        p = Path(p)
        key = str(p)
        if key not in seen and p.exists():
            uniq.append(p)
            seen.add(key)
    return uniq

def resolve_cache_path(cache_name):
    cache_name = Path(cache_name).name
    for root in _get_cache_search_dirs():
        p = root / cache_name
        if p.exists():
            return p
    return CACHE_DIR / cache_name

def cache_exists(cache_name):
    return resolve_cache_path(cache_name).exists()

def maybe_load_cache(cache_name, force_rebuild=False):
    if force_rebuild or (not USE_CACHE):
        return None

    path = resolve_cache_path(cache_name)
    if path.exists():
        print(f"[cache] using: {path}")
        return np.load(path, allow_pickle=True)

    return None

def parallel_map_targets(worker_fn, n_targets, max_workers=None, desc=None, progress_step=5):
    if max_workers is None:
        max_workers = PARALLEL_TARGET_WORKERS
    max_workers = max(1, min(max_workers, n_targets))

    if max_workers == 1:
        out = []
        for j in range(n_targets):
            out.append(worker_fn(j))
            if desc and (((j + 1) % progress_step == 0) or (j + 1 == n_targets)):
                print(f"{desc}: {j+1:02d}/{n_targets}")
        return out

    results = [None] * n_targets
    done = 0
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        fut_to_j = {ex.submit(worker_fn, j): j for j in range(n_targets)}
        for fut in as_completed(fut_to_j):
            j = fut_to_j[fut]
            results[j] = fut.result()
            done += 1
            if desc and ((done % progress_step == 0) or (done == n_targets)):
                print(f"{desc}: {done:02d}/{n_targets}")
    return results

def safe_auc(y_true, y_pred):
    y_true = np.asarray(y_true)
    if np.unique(y_true).size < 2:
        return 0.5
    return float(roc_auc_score(y_true, y_pred))

def macro_auc(y_true, y_pred, verbose=False):
    n_targets = y_true.shape[1]
    def worker(j):
        return safe_auc(y_true[:, j], y_pred[:, j])
    aucs = parallel_map_targets(worker, n_targets=n_targets, max_workers=PARALLEL_TARGET_WORKERS)
    score = float(np.mean(aucs))
    if verbose:
        for name, auc in zip(target_cols, aucs):
            print(f"{name:20s} {auc:.6f}")
    return score

def percentile_ranks(a, max_workers=None):
    a = np.asarray(a)
    n, m = a.shape
    out = np.empty((n, m), dtype=np.float32)
    base = (np.arange(n, dtype=np.float32) + 0.5) / n

    def worker(j):
        idx = np.argsort(a[:, j], kind="mergesort")
        return j, idx

    results = parallel_map_targets(
        worker,
        n_targets=m,
        max_workers=max_workers if max_workers is not None else PARALLEL_TARGET_WORKERS,
    )
    for j, idx in results:
        out[idx, j] = base
    return out

def print_scores_table(score_dict, title):
    rows = sorted(score_dict.items(), key=lambda x: -x[1])
    print("\n" + "=" * 96)
    print(title)
    print("=" * 96)
    for name, score in rows:
        print(f"{name:36s} {score:.6f}")

def build_meta_features(
    pred_list,
    rank_list=None,
    include_raw=True,
    include_ranks=True,
    include_pairwise_abs=True,
    include_pairwise_prod=False,
    include_stats=True,
):
    pred_list = [np.asarray(x, dtype=np.float32) for x in pred_list]
    if rank_list is None:
        rank_list = [percentile_ranks(x) for x in pred_list]
    else:
        rank_list = [np.asarray(x, dtype=np.float32) for x in rank_list]

    feats = []
    if include_raw:
        feats.extend(pred_list)
    if include_ranks:
        feats.extend(rank_list)
    if include_pairwise_abs:
        for i, j in combinations(range(len(pred_list)), 2):
            feats.append(np.abs(pred_list[i] - pred_list[j]).astype(np.float32))
    if include_pairwise_prod:
        for i, j in combinations(range(len(pred_list)), 2):
            feats.append((pred_list[i] * pred_list[j]).astype(np.float32))
    if include_stats:
        stacked = np.stack(pred_list, axis=2)
        feats.append(stacked.mean(axis=2).astype(np.float32))
        feats.append(stacked.std(axis=2).astype(np.float32))
        feats.append(stacked.min(axis=2).astype(np.float32))
        feats.append(stacked.max(axis=2).astype(np.float32))

    return np.hstack(feats).astype(np.float32)

def normalize_rows_if_needed(w):
    w = np.asarray(w, dtype=np.float32)
    row_sum = w.sum(axis=1, keepdims=True)
    bad = row_sum[:, 0] <= 0
    if np.any(bad):
        w[bad] = 1.0 / w.shape[1]
        row_sum = w.sum(axis=1, keepdims=True)
    w = w / row_sum
    return w.astype(np.float32)

def apply_weight_matrix(rank_dict, model_names, weights, row_idx=None):
    first = rank_dict[model_names[0]]
    n_rows = first.shape[0] if row_idx is None else len(row_idx)
    n_targets = weights.shape[0]
    out = np.zeros((n_rows, n_targets), dtype=np.float32)
    for i, name in enumerate(model_names):
        arr = rank_dict[name] if row_idx is None else rank_dict[name][row_idx]
        out += arr * weights[:, i][None, :]
    return out.astype(np.float32)

def mean_rank_correlation_matrix(pred_dict, model_names):
    ranks = {name: percentile_ranks(pred_dict[name]) for name in model_names}
    corr = np.eye(len(model_names), dtype=np.float32)
    for i, a in enumerate(model_names):
        xa = ranks[a].reshape(-1)
        for j, b in enumerate(model_names):
            if j <= i:
                continue
            xb = ranks[b].reshape(-1)
            c = float(np.corrcoef(xa, xb)[0, 1])
            corr[i, j] = c
            corr[j, i] = c
    return pd.DataFrame(corr, index=model_names, columns=model_names)

def get_folds(n_rows, n_splits=5, seed=42):
    return list(KFold(n_splits=n_splits, shuffle=True, random_state=seed).split(np.arange(n_rows)))

def save_submission(test_pred, filename):
    out_path = SUB_DIR / filename
    sub = pl.DataFrame({"customer_id": sample_submit["customer_id"]}).hstack(
        pl.DataFrame(test_pred.astype(np.float64), schema=predict_cols)
    )
    sub.write_parquet(out_path)
    print("Saved:", out_path)

# ---------- load to pandas ----------
train_df = train_feat_pl.to_pandas()
test_df = test_feat_pl.to_pandas()
targets_df = targets_pl.to_pandas()

y = targets_df[target_cols].to_numpy(dtype=np.float32)
n_train, n_targets = y.shape
n_test = len(test_df)

print("n_train:", n_train, "| n_test:", n_test, "| n_targets:", n_targets)

# ---------- encode categoricals jointly ----------
def joint_encode_categories(train_df, test_df, cat_cols):
    train_cat = np.zeros((len(train_df), len(cat_cols)), dtype=np.int32)
    test_cat = np.zeros((len(test_df), len(cat_cols)), dtype=np.int32)
    cat_cardinalities = []

    for i, col in enumerate(cat_cols):
        tr_s = train_df[col]
        te_s = test_df[col]
        combined = pd.concat([tr_s, te_s], axis=0, ignore_index=True)

        missing_mask = combined.isna().to_numpy()
        combined_str = combined.astype("string").fillna("__MISSING__")
        codes, uniques = pd.factorize(combined_str, sort=False)
        codes = codes.astype(np.int32) + 1
        codes[missing_mask] = 0

        train_cat[:, i] = codes[: len(train_df)]
        test_cat[:, i] = codes[len(train_df):]
        cat_cardinalities.append(int(codes.max()) + 1)

    return train_cat, test_cat, cat_cardinalities

train_cat_np, test_cat_np, cat_cardinalities = joint_encode_categories(train_df, test_df, cat_cols)

train_num_np = train_df[num_cols].to_numpy(dtype=np.float32)
test_num_np = test_df[num_cols].to_numpy(dtype=np.float32)

train_tree_np = np.hstack([train_num_np, train_cat_np.astype(np.float32)]).astype(np.float32)
test_tree_np = np.hstack([test_num_np, test_cat_np.astype(np.float32)]).astype(np.float32)

tree_feature_names = num_cols + cat_cols
cat_feature_indices = list(range(len(num_cols), len(tree_feature_names)))

train_catboost_df = pd.DataFrame(
    np.hstack([train_num_np, train_cat_np.astype(np.float32)]),
    columns=tree_feature_names,
)
test_catboost_df = pd.DataFrame(
    np.hstack([test_num_np, test_cat_np.astype(np.float32)]),
    columns=tree_feature_names,
)
for c in cat_cols:
    train_catboost_df[c] = train_catboost_df[c].astype(np.int32)
    test_catboost_df[c] = test_catboost_df[c].astype(np.int32)

base_folds = get_folds(n_train, n_splits=N_FOLDS_BASE, seed=SEED)

print("train_num_np:", train_num_np.shape)
print("train_cat_np:", train_cat_np.shape)
print("train_tree_np:", train_tree_np.shape)
print("cat cardinalities example:", cat_cardinalities[:5])

# ---------- external checkpoint helpers ----------
def load_checkpoint_oof_test(path, model_name="model"):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{model_name} checkpoint not found: {path}")
    data = np.load(path, allow_pickle=True)
    keys = set(data.files)
    candidate_pairs = [
        ("oof_preds", "test_preds"),
        ("oof", "test"),
        ("oof_lgbm", "test_lgbm"),
        ("oof_lgb", "test_lgb"),
        ("oof_pb", "test_pb"),
        ("oof_pyboost", "test_pyboost"),
        ("val_preds", "preds_test"),
    ]
    for oof_key, test_key in candidate_pairs:
        if oof_key in keys and test_key in keys:
            oof_pred = data[oof_key].astype(np.float32)
            test_pred = data[test_key].astype(np.float32)
            print(f"[{model_name}] loaded from {path}")
            print(f"[{model_name}] keys: {oof_key}, {test_key}")
            print(f"[{model_name}] shapes: oof={oof_pred.shape}, test={test_pred.shape}")
            return oof_pred, test_pred
    raise KeyError(f"Could not find supported OOF/test keys in {path}. Available keys: {sorted(keys)}")

def validate_prediction_shapes(oof_pred, test_pred, n_train, n_test, n_targets, model_name):
    if oof_pred.ndim != 2 or oof_pred.shape != (n_train, n_targets):
        raise ValueError(f"{model_name} OOF shape mismatch: got {oof_pred.shape}, expected {(n_train, n_targets)}")
    if test_pred.ndim != 2 or test_pred.shape != (n_test, n_targets):
        raise ValueError(f"{model_name} TEST shape mismatch: got {test_pred.shape}, expected {(n_test, n_targets)}")

def try_hydrate_cached_model(model_name, cache_name, oof_dict, test_dict):
    path = resolve_cache_path(cache_name)
    if not path.exists():
        return False

    data = np.load(path, allow_pickle=True)
    keys = set(data.files)

    if "oof_preds" not in keys or "test_preds" not in keys:
        print(f"[cache] skip {model_name}: no oof_preds/test_preds in {path}")
        return False

    oof_pred = data["oof_preds"].astype(np.float32)
    test_pred = data["test_preds"].astype(np.float32)

    validate_prediction_shapes(
        oof_pred, test_pred,
        n_train=n_train, n_test=n_test, n_targets=n_targets,
        model_name=model_name,
    )

    oof_dict[model_name] = oof_pred
    test_dict[model_name] = test_pred
    print(f"[cache] hydrated {model_name} <- {path}")
    return True

def preload_available_cached_models(oof_dict, test_dict):
    registry = [
        ("blend3_cf", "blend3_cf.npz"),
        ("rank_sparse_cf", "rank_sparse_cf.npz"),

        ("ridge_meta_l1", "ridge_meta_l1.npz"),
        ("lgb_meta_l1", "lgb_meta_l1.npz"),
        ("xgb_meta_l1", "xgb_meta_l1.npz"),
        ("cat_meta_l1", "cat_meta_l1.npz"),
        ("pyboost_meta_l1", "pyboost_meta_l1.npz"),

        ("ridge_meta_l2", "ridge_meta_l2.npz"),
        ("lgb_meta_l2", "lgb_meta_l2.npz"),
        ("xgb_meta_l2", "xgb_meta_l2.npz"),
        ("cat_meta_l2", "cat_meta_l2.npz"),
        ("pyboost_meta_l2", "pyboost_meta_l2.npz"),

        ("ridge_meta_l3", "ridge_meta_l3.npz"),
        ("lgb_meta_l3", "lgb_meta_l3.npz"),
        ("xgb_meta_l3", "xgb_meta_l3.npz"),
        ("cat_meta_l3", "cat_meta_l3.npz"),
        ("pyboost_meta_l3", "pyboost_meta_l3.npz"),

        ("rank_sparse_l2_cf", "rank_sparse_l2_cf.npz"),
        ("rank_sparse_l3_cf", "rank_sparse_l3_cf.npz"),
        ("superblend_cf", "superblend_cf.npz"),
    ]

    loaded = []
    for model_name, cache_name in registry:
        try:
            ok = try_hydrate_cached_model(model_name, cache_name, oof_dict, test_dict)
            if ok:
                loaded.append(model_name)
        except Exception as e:
            print(f"[cache] failed {model_name} from {cache_name}: {repr(e)}")

    print("\nPreloaded cached models:", loaded if loaded else "none")
def try_load_prediction_checkpoint(path, model_name, n_train, n_test, n_targets):
    path = Path(path)
    if not path.exists():
        print(f"[{model_name}] checkpoint not found: {path}")
        return None

    try:
        oof_pred, test_pred = load_checkpoint_oof_test(path, model_name=model_name)
        validate_prediction_shapes(
            oof_pred, test_pred,
            n_train=n_train, n_test=n_test, n_targets=n_targets,
            model_name=model_name,
        )
        print(f"[{model_name}] valid checkpoint found -> skip training")
        return oof_pred.astype(np.float32), test_pred.astype(np.float32)
    except Exception as e:
        print(f"[{model_name}] checkpoint exists but is invalid: {path}")
        print(f"[{model_name}] reason: {repr(e)}")
        return None

def save_prediction_checkpoint(path, model_name, oof_pred, test_pred, extra_meta=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    meta = {
        "model_name": model_name,
        "oof_shape": list(np.asarray(oof_pred).shape),
        "test_shape": list(np.asarray(test_pred).shape),
    }
    if extra_meta is not None:
        meta.update(extra_meta)

    np.savez_compressed(
        path,
        oof_preds=np.asarray(oof_pred, dtype=np.float32),
        test_preds=np.asarray(test_pred, dtype=np.float32),
        meta_json=np.array(json.dumps(meta, ensure_ascii=False)),
    )
    print(f"[{model_name}] checkpoint saved -> {path}")

def print_internal_checkpoint_status(n_train, n_test, n_targets):
    registry = {
        "nn_base": NN_BASE_PRED_CKPT_PATH,
        "xgb_base": XGB_BASE_PRED_CKPT_PATH,
        "cat_base": CAT_BASE_PRED_CKPT_PATH,
    }
    print("\nInternal base-model checkpoint status:")
    for name, path in registry.items():
        if not Path(path).exists():
            print(f"  {name:10s} -> MISSING | {path}")
            continue
        try:
            oof_pred, test_pred = load_checkpoint_oof_test(path, model_name=name)
            validate_prediction_shapes(oof_pred, test_pred, n_train, n_test, n_targets, name)
            print(f"  {name:10s} -> OK      | {path}")
        except Exception as e:
            print(f"  {name:10s} -> INVALID | {path} | {repr(e)}")


n_train: 750000 | n_test: 250000 | n_targets: 41
train_num_np: (750000, 2199)
train_cat_np: (750000, 61)
train_tree_np: (750000, 2260)
cat cardinalities example: [3, 4, 4, 3, 4]


In [5]:

# ============================================================
# 4) DEVICE POLICY / DETECTION (UPDATED)
# ============================================================
import os
import sys
import subprocess
import warnings
import numpy as np
import torch

warnings.filterwarnings("ignore")

if "CPU_COUNT" not in globals():
    CPU_COUNT = max(1, os.cpu_count() or 1)
if "FORCE_NN_CPU" not in globals():
    FORCE_NN_CPU = False
if "FORCE_XGB_GPU" not in globals():
    FORCE_XGB_GPU = True
if "FORCE_CAT_GPU" not in globals():
    FORCE_CAT_GPU = True
if "ENABLE_PYBOOST_META" not in globals():
    ENABLE_PYBOOST_META = True
if "FORCE_PYBOOST_META_GPU" not in globals():
    FORCE_PYBOOST_META_GPU = True
if "INSTALL_PYBOOST_META" not in globals():
    INSTALL_PYBOOST_META = True
if "CUPY_PACKAGE" not in globals():
    CUPY_PACKAGE = "cupy-cuda12x"

# CPU thread policy for CPU-bound parts (feature engineering / LGB meta / misc)
os.environ["OMP_NUM_THREADS"] = str(CPU_COUNT)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_COUNT)
os.environ["MKL_NUM_THREADS"] = str(CPU_COUNT)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_COUNT)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_COUNT)

def _has_cuda_runtime():
    try:
        return bool(torch.cuda.is_available())
    except Exception:
        return False

def _has_mps_runtime():
    try:
        return bool(torch.backends.mps.is_available())
    except Exception:
        return False

def _pip_install_quiet(packages):
    if isinstance(packages, str):
        packages = [packages]
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(packages))
        return True, None
    except Exception as e:
        return False, repr(e)

def smoke_test_xgb_cuda():
    try:
        import xgboost as xgb
        X = np.random.randn(512, 8).astype(np.float32)
        y_ = (X[:, 0] > 0).astype(np.int32)
        model = xgb.XGBClassifier(
            objective="binary:logistic",
            eval_metric="auc",
            n_estimators=20,
            max_depth=3,
            tree_method="hist",
            device="cuda",
            verbosity=0,
        )
        model.fit(X, y_, verbose=False)
        _ = model.predict_proba(X[:32])
        return True, None
    except Exception as e:
        return False, repr(e)

def smoke_test_cat_cuda():
    try:
        from catboost import CatBoostClassifier
        X = np.random.randn(512, 8).astype(np.float32)
        y_ = (X[:, 0] > 0).astype(np.int32)
        model = CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            iterations=20,
            depth=3,
            learning_rate=0.1,
            task_type="GPU",
            devices="0",
            verbose=False,
            allow_writing_files=False,
        )
        model.fit(X, y_)
        _ = model.predict_proba(X[:32])
        return True, None
    except Exception as e:
        return False, repr(e)

def detect_pyboost_gpu_ready():
    try:
        import cupy  # noqa: F401
        from py_boost import GradientBoosting  # noqa: F401
        return True, None
    except Exception as e:
        return False, repr(e)

def maybe_install_pyboost_meta():
    ok, err = detect_pyboost_gpu_ready()
    if ok:
        return True, None
    if not INSTALL_PYBOOST_META:
        return False, err
    print("[PyBoost-meta] py_boost/cupy not found. Trying installation...")
    ok_install, err_install = _pip_install_quiet([CUPY_PACKAGE, "py-boost"])
    if not ok_install:
        return False, f"install failed: {err_install}"
    return detect_pyboost_gpu_ready()

# NN device should follow the uploaded script behavior (prefer GPU, then MPS, then CPU)
if FORCE_NN_CPU:
    TORCH_DEVICE = torch.device("cpu")
else:
    if _has_cuda_runtime():
        TORCH_DEVICE = torch.device("cuda")
    elif _has_mps_runtime():
        TORCH_DEVICE = torch.device("mps")
    else:
        TORCH_DEVICE = torch.device("cpu")

# Threads still matter for CPU fallback and preprocessing
try:
    torch.set_num_threads(CPU_COUNT)
except Exception:
    pass
try:
    torch.set_num_interop_threads(max(1, min(8, CPU_COUNT)))
except Exception:
    pass

# LGBM strictly on CPU
LGB_DEVICE = "cpu"

# XGBoost strictly on GPU
if FORCE_XGB_GPU:
    if not _has_cuda_runtime():
        raise RuntimeError("FORCE_XGB_GPU=True, but CUDA GPU is not available.")
    _ok_xgb, _err_xgb = smoke_test_xgb_cuda()
    if not _ok_xgb:
        raise RuntimeError(f"XGBoost GPU smoke-test failed: {_err_xgb}")
    XGB_DEVICE = "cuda"
else:
    XGB_DEVICE = "cpu"

# CatBoost strictly on GPU
if FORCE_CAT_GPU:
    if not _has_cuda_runtime():
        raise RuntimeError("FORCE_CAT_GPU=True, but CUDA GPU is not available.")
    _ok_cat, _err_cat = smoke_test_cat_cuda()
    if not _ok_cat:
        raise RuntimeError(f"CatBoost GPU smoke-test failed: {_err_cat}")
    CAT_DEVICE = "gpu"
else:
    CAT_DEVICE = "cpu"

# PyBoost meta on GPU; try install automatically if missing
PYBOOST_META_READY = False
PYBOOST_META_ERROR = "disabled"
PYBOOST_META_DEVICE = None
if ENABLE_PYBOOST_META:
    if FORCE_PYBOOST_META_GPU and not _has_cuda_runtime():
        print("[PyBoost-meta] CUDA GPU is not available -> disabling PyBoost meta.")
        ENABLE_PYBOOST_META = False
        PYBOOST_META_ERROR = "CUDA unavailable"
    else:
        ok_pb, err_pb = maybe_install_pyboost_meta()
        if ok_pb:
            PYBOOST_META_READY = True
            PYBOOST_META_ERROR = None
            PYBOOST_META_DEVICE = "gpu"
        else:
            ENABLE_PYBOOST_META = False
            PYBOOST_META_READY = False
            PYBOOST_META_ERROR = err_pb
            print("[PyBoost-meta] py_boost/cupy unavailable -> disabling PyBoost meta.")
            print("[PyBoost-meta] reason:", PYBOOST_META_ERROR)

print("TORCH_DEVICE (NN):", TORCH_DEVICE)
print("LGB_DEVICE       :", LGB_DEVICE)
print("XGB_DEVICE       :", XGB_DEVICE)
print("CAT_DEVICE       :", CAT_DEVICE)
print("PYBOOST_META     :", ENABLE_PYBOOST_META, "| ready =", PYBOOST_META_READY, "|", PYBOOST_META_ERROR)
print("CPU threads      :", CPU_COUNT)


Default metric period is 5 because AUC is/are not implemented for GPU


TORCH_DEVICE (NN): cuda
LGB_DEVICE       : cpu
XGB_DEVICE       : cuda
CAT_DEVICE       : gpu
PYBOOST_META     : True | ready = True | None
CPU threads      : 26


In [6]:

# ============================================================
# 5) BASE MODEL: NN (best uploaded version, adapted to notebook)
# ============================================================
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import QuantileTransformer

class DenoisingAutoencoder(nn.Module):
    """Symmetric DAE: Input → 1024 → 512 → bottleneck → 512 → 1024 → Input."""
    def __init__(self, input_dim, bottleneck_dim=DAE_BOTTLENECK_DIM):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 1024), nn.BatchNorm1d(1024), nn.SiLU(), nn.Dropout(0.3),
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(512, bottleneck_dim), nn.BatchNorm1d(bottleneck_dim), nn.SiLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, 512), nn.BatchNorm1d(512), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(512, 1024), nn.BatchNorm1d(1024), nn.SiLU(), nn.Dropout(0.3),
            nn.Linear(1024, input_dim),
        )
        for module in [self.encoder, self.decoder]:
            for layer in module:
                if isinstance(layer, nn.Linear):
                    nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
                    nn.init.zeros_(layer.bias)

    def encode(self, x):
        return self.encoder(x)

    def forward(self, x):
        return self.decoder(self.encoder(x))


def apply_swap_noise(clean, non_null_mask, corruption_rate):
    corrupt_mask = (torch.rand_like(clean) < corruption_rate) & non_null_mask
    perm = torch.randperm(clean.shape[0], device=clean.device)
    return torch.where(corrupt_mask, clean[perm], clean)


def train_dae_on_numpy(all_num_data, null_mask_np, device, fold_dir):
    n_samples, input_dim = all_num_data.shape
    print(f"    DAE: {n_samples:,} × {input_dim}, bottleneck={DAE_BOTTLENECK_DIM}")

    model = DenoisingAutoencoder(input_dim, DAE_BOTTLENECK_DIM).to(device)
    use_compile = hasattr(torch, "compile") and device.type != "cpu"
    if use_compile:
        try:
            backend = "aot_eager" if device.type == "mps" else "inductor"
            model = torch.compile(model, backend=backend)
        except Exception:
            use_compile = False

    optimizer = torch.optim.AdamW(model.parameters(), lr=DAE_LR, weight_decay=DAE_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=DAE_EPOCHS)

    data_tensor = torch.from_numpy(all_num_data).to(device)
    non_null_tensor = torch.from_numpy((1 - null_mask_np).astype(np.float32)).to(device)
    del null_mask_np
    gc.collect()

    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)
    n_batches = (n_samples + DAE_BATCH_SIZE - 1) // DAE_BATCH_SIZE

    fold_dir.mkdir(parents=True, exist_ok=True)
    ckpt_final = fold_dir / "dae_final.pt"
    if ckpt_final.exists():
        print(f"    Loading DAE from {ckpt_final}")
        ckpt = torch.load(ckpt_final, map_location=device)
        orig_model = model._orig_mod if use_compile and hasattr(model, "_orig_mod") else model
        orig_model.load_state_dict(ckpt["model_state_dict"])
        del data_tensor, non_null_tensor, ckpt
        return model

    for epoch in range(DAE_EPOCHS):
        model.train()
        total_loss = 0.0
        perm_indices = torch.randperm(n_samples, device=device)
        for b in range(n_batches):
            start = b * DAE_BATCH_SIZE
            end = min(start + DAE_BATCH_SIZE, n_samples)
            idx = perm_indices[start:end]
            clean = data_tensor[idx]
            non_null = non_null_tensor[idx]

            with torch.autocast(device_type=device.type if device.type != "mps" else "cpu", dtype=torch.float16, enabled=use_amp):
                corrupted = apply_swap_noise(clean, non_null > 0.5, DAE_CORRUPTION_RATE)
                reconstructed = model(corrupted)
                diff_sq = (reconstructed.float() - clean.float()) ** 2 * non_null
                loss = diff_sq.sum() / non_null.sum().clamp(min=1)

            optimizer.zero_grad()
            if use_amp:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), DAE_GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), DAE_GRAD_CLIP)
                optimizer.step()
            total_loss += float(loss.item())

        scheduler.step()
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"      Epoch {epoch+1:2d}/{DAE_EPOCHS}  loss={total_loss/n_batches:.6f}", flush=True)

    orig_model = model._orig_mod if use_compile and hasattr(model, "_orig_mod") else model
    torch.save({
        "epoch": DAE_EPOCHS,
        "model_state_dict": orig_model.state_dict(),
        "loss": total_loss / n_batches,
    }, ckpt_final)
    del data_tensor, non_null_tensor
    return model


@torch.no_grad()
def extract_dae_embeddings(model, all_num_data, device, batch_size=4096):
    model.eval()
    embeddings = []
    for start in range(0, all_num_data.shape[0], batch_size):
        end = min(start + batch_size, all_num_data.shape[0])
        batch = torch.from_numpy(all_num_data[start:end]).to(device)
        out = model.encode(batch)
        embeddings.append(out.detach().cpu().numpy())
    return np.vstack(embeddings).astype(np.float32)


class LinearBE(nn.Module):
    def __init__(self, in_dim, out_dim, k):
        super().__init__()
        self.k = k
        self.linear = nn.Linear(in_dim, out_dim, bias=False)
        self.r = nn.Parameter(torch.ones(k, in_dim))
        self.s = nn.Parameter(torch.ones(k, out_dim))
        self.b = nn.Parameter(torch.zeros(k, out_dim))

    def forward(self, x):
        x = x * self.r.unsqueeze(0)
        B, k, D = x.shape
        x = self.linear(x.reshape(B * k, D)).reshape(B, k, -1)
        return x * self.s.unsqueeze(0) + self.b.unsqueeze(0)


class ResidualBlockBE(nn.Module):
    def __init__(self, dim, k, dropout=0.3):
        super().__init__()
        self.bn1 = nn.BatchNorm1d(dim)
        self.lin1 = LinearBE(dim, dim, k)
        self.bn2 = nn.BatchNorm1d(dim)
        self.lin2 = LinearBE(dim, dim, k)
        self.drop = nn.Dropout(dropout)
        self.act = nn.SiLU()
        nn.init.zeros_(self.lin2.linear.weight)

    def forward(self, x):
        B, k, D = x.shape
        h = self.bn1(x.reshape(B * k, D)).reshape(B, k, D)
        h = self.act(h)
        h = self.lin1(h)
        B, k, D = h.shape
        h = self.bn2(h.reshape(B * k, D)).reshape(B, k, D)
        h = self.act(h)
        h = self.drop(h)
        h = self.lin2(h)
        return x + h


class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=ASL_GAMMA_NEG, gamma_pos=ASL_GAMMA_POS, clip=ASL_CLIP, eps=1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, logits, targets):
        p = torch.sigmoid(logits)
        pos_term = (1 - p).pow(self.gamma_pos) * torch.log(p.clamp(min=self.eps))
        p_m = (p - self.clip).clamp(min=self.eps) if self.clip > 0 else p
        neg_term = p_m.pow(self.gamma_neg) * torch.log((1 - p_m).clamp(min=self.eps))
        return (-(targets * pos_term) - (1 - targets) * neg_term).mean()


class PiecewiseLinearEncoding(nn.Module):
    def __init__(self, n_features, n_bins=PLR_N_BINS):
        super().__init__()
        self.n_features = n_features
        self.n_bins = n_bins
        self.register_buffer('edges', torch.zeros(n_features, n_bins + 1))
        self.weight = nn.Parameter(torch.empty(n_features, n_bins))
        self.bias = nn.Parameter(torch.zeros(n_features))
        nn.init.kaiming_uniform_(self.weight, a=5**0.5)

    def set_bins(self, X_train_np):
        quantiles = np.linspace(0, 1, self.n_bins + 1)
        for i in range(self.n_features):
            col = X_train_np[:, i]
            col_valid = col[~np.isnan(col)] if np.any(np.isnan(col)) else col
            if len(col_valid) < 2:
                edges = np.linspace(-3.0, 3.0, self.n_bins + 1)
            else:
                edges = np.unique(np.quantile(col_valid, quantiles))
                if len(edges) < self.n_bins + 1:
                    edges = np.linspace(edges[0], edges[-1], self.n_bins + 1)
                else:
                    edges = edges[:self.n_bins + 1]
            self.edges[i] = torch.from_numpy(edges.astype(np.float32))

    def forward(self, x):
        left = self.edges[:, :-1]
        right = self.edges[:, 1:]
        width = (right - left).clamp(min=1e-8)
        ple = ((x.unsqueeze(2) - left) / width).clamp(0, 1)
        return (ple * self.weight).sum(dim=2) + self.bias


class TabularNet(nn.Module):
    def __init__(self, cat_cardinalities, n_numerical, n_targets, k=K_ENSEMBLE):
        super().__init__()
        self.k = k
        self.n_targets = n_targets
        self.plr = PiecewiseLinearEncoding(n_numerical, PLR_N_BINS)

        self.embeddings = nn.ModuleList()
        total_embed_dim = 0
        for n_cat in cat_cardinalities:
            dim = min(50, max(2, (n_cat + 1) // 2))
            self.embeddings.append(nn.Embedding(n_cat + 2, dim))
            total_embed_dim += dim

        input_dim = total_embed_dim + n_numerical
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, NN_HIDDEN_DIM), nn.BatchNorm1d(NN_HIDDEN_DIM), nn.SiLU(),
        )
        self.res_blocks = nn.ModuleList([
            ResidualBlockBE(NN_HIDDEN_DIM, k, dropout=0.4),
            ResidualBlockBE(NN_HIDDEN_DIM, k, dropout=0.4),
            ResidualBlockBE(NN_HIDDEN_DIM, k, dropout=0.3),
        ])
        head_dim = NN_HIDDEN_DIM // 2
        self.head_bn1 = nn.BatchNorm1d(NN_HIDDEN_DIM)
        self.head_drop1 = nn.Dropout(0.2)
        self.head_lin1 = LinearBE(NN_HIDDEN_DIM, head_dim, k)
        self.head_bn2 = nn.BatchNorm1d(head_dim)
        self.head_drop2 = nn.Dropout(0.1)
        self.head_lin2 = LinearBE(head_dim, n_targets, k)
        self.act = nn.SiLU()

        for layer in self.input_proj:
            if isinstance(layer, nn.Linear):
                nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
                nn.init.zeros_(layer.bias)
        nn.init.kaiming_normal_(self.head_lin1.linear.weight, nonlinearity="relu")
        nn.init.kaiming_normal_(self.head_lin2.linear.weight, nonlinearity="relu")

    def _embed_and_project(self, x_cat, x_num):
        x_num = self.plr(x_num)
        embeds = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        return self.input_proj(torch.cat(embeds + [x_num], dim=1))

    def _head(self, x):
        B, k, D = x.shape
        x = self.head_bn1(x.reshape(B * k, D)).reshape(B, k, D)
        x = self.act(x)
        x = self.head_drop1(x)
        x = self.head_lin1(x)
        B, k, D2 = x.shape
        x = self.head_bn2(x.reshape(B * k, D2)).reshape(B, k, D2)
        x = self.act(x)
        x = self.head_drop2(x)
        return self.head_lin2(x)

    def forward(self, x_cat, x_num):
        x = self._embed_and_project(x_cat, x_num).unsqueeze(1).expand(-1, self.k, -1)
        for block in self.res_blocks:
            x = block(x)
        logits = self._head(x)
        if self.training:
            return logits
        return torch.sigmoid(logits).mean(dim=1)

    def forward_mixup(self, x_cat, x_num, lam, perm, mixup_layer):
        x = self._embed_and_project(x_cat, x_num).unsqueeze(1).expand(-1, self.k, -1)
        if mixup_layer == 0:
            x = lam * x + (1 - lam) * x[perm]
        for i, block in enumerate(self.res_blocks):
            x = block(x)
            if mixup_layer == i + 1:
                x = lam * x + (1 - lam) * x[perm]
        return self._head(x)


def quantile_normalize_np(train_num, val_num, test_num):
    n = train_num.shape[0]
    qt = QuantileTransformer(
        n_quantiles=min(1000, n),
        output_distribution="normal",
        subsample=min(100_000, n),
        random_state=SEED,
    )
    qt.fit(train_num)
    tr = np.nan_to_num(qt.transform(train_num), nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    va = np.nan_to_num(qt.transform(val_num), nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    te = np.nan_to_num(qt.transform(test_num), nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return tr, va, te


def train_epoch_tabm(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, n_batches = 0.0, 0
    k = model.k
    for x_cat, x_num, y in loader:
        x_cat = x_cat.to(device)
        x_num = x_num.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        mixup_layer = np.random.randint(0, 4)
        lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
        perm = torch.randperm(x_cat.shape[0], device=device)
        logits = model.forward_mixup(x_cat, x_num, lam, perm, mixup_layer)
        y_k = y.unsqueeze(1).expand(-1, k, -1)
        y_mixed = lam * y_k + (1 - lam) * y[perm].unsqueeze(1).expand(-1, k, -1)
        loss = criterion(logits, y_mixed)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), NN_GRAD_CLIP)
        optimizer.step()
        total_loss += float(loss.item())
        n_batches += 1
    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate_tabm(model, loader, device):
    model.eval()
    all_preds, all_targets = [], []
    for x_cat, x_num, y in loader:
        probs = model(x_cat.to(device), x_num.to(device))
        all_preds.append(probs.cpu())
        all_targets.append(y)
    return torch.cat(all_preds).numpy().astype(np.float32), torch.cat(all_targets).numpy().astype(np.float32)


@torch.no_grad()
def predict_tabm(model, loader, device):
    model.eval()
    all_preds = []
    for batch in loader:
        probs = model(batch[0].to(device), batch[1].to(device))
        all_preds.append(probs.cpu())
    return torch.cat(all_preds).numpy().astype(np.float32)


def train_one_fold_best_nn(
    fold_idx,
    tr_cat,
    tr_num,
    tr_y,
    val_cat,
    val_num,
    val_y,
    test_cat,
    test_num,
    cat_cardinalities,
    target_cols,
    device,
):
    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(tr_cat), torch.from_numpy(tr_num), torch.from_numpy(tr_y)),
        batch_size=NN_BATCH_SIZE,
        shuffle=True,
        drop_last=True,
    )
    val_loader = DataLoader(
        TensorDataset(torch.from_numpy(val_cat), torch.from_numpy(val_num), torch.from_numpy(val_y)),
        batch_size=NN_BATCH_SIZE * 2,
        shuffle=False,
    )
    test_loader = DataLoader(
        TensorDataset(torch.from_numpy(test_cat), torch.from_numpy(test_num)),
        batch_size=NN_BATCH_SIZE * 2,
        shuffle=False,
    )

    model = TabularNet(cat_cardinalities, tr_num.shape[1], len(target_cols)).to(device)
    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters())
        print(f"  NN model: {n_params:,} parameters (k={K_ENSEMBLE})")

    model.plr.set_bins(tr_num)
    criterion = AsymmetricLoss(ASL_GAMMA_NEG, ASL_GAMMA_POS, ASL_CLIP)
    optimizer = torch.optim.AdamW(model.parameters(), lr=NN_LR, weight_decay=NN_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

    best_auc, patience_counter = -1.0, 0
    top_states = []

    for epoch in range(NN_EPOCHS):
        t1 = time.time()
        train_loss = train_epoch_tabm(model, train_loader, optimizer, criterion, device)
        val_preds, val_targets = evaluate_tabm(model, val_loader, device)
        val_macro_auc = macro_auc(val_targets, val_preds)
        scheduler.step(val_macro_auc)

        improved = ""
        if val_macro_auc > best_auc + 1e-6:
            best_auc = val_macro_auc
            patience_counter = 0
            improved = " *"
        else:
            patience_counter += 1

        state_copy = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        top_states.append((val_macro_auc, state_copy))
        top_states.sort(key=lambda x: x[0], reverse=True)
        if len(top_states) > TOP_K_SWA:
            top_states.pop()

        print(
            f"    Ep {epoch+1:2d}/{NN_EPOCHS}  loss={train_loss:.4f}  val_auc={val_macro_auc:.4f}  "
            f"lr={optimizer.param_groups[0]['lr']:.1e}  {time.time()-t1:.1f}s{improved}",
            flush=True,
        )

        if patience_counter >= NN_PATIENCE:
            print(f"    Early stop at epoch {epoch+1}")
            break

    avg_state = {}
    for key in top_states[0][1]:
        avg_state[key] = sum(s[key] for _, s in top_states) / len(top_states)
    model.load_state_dict(avg_state)
    model.to(device)

    val_preds, _ = evaluate_tabm(model, val_loader, device)
    swa_auc = macro_auc(val_y, val_preds)
    test_preds = predict_tabm(model, test_loader, device)

    print(f"  Fold {fold_idx+1} best={best_auc:.4f}, SWA={swa_auc:.4f}", flush=True)
    return val_preds.astype(np.float32), test_preds.astype(np.float32), float(swa_auc)


def fit_nn_base_best(
    train_num_full,
    test_num_full,
    train_cat_full,
    test_cat_full,
    y_train,
    num_feature_names,
    target_cols,
    cat_cardinalities,
    folds,
    cache_name="base_nn_best.npz",
):
    fold_dir = NN_CKPT_DIR
    device = TORCH_DEVICE
    n_train, n_targets = y_train.shape
    n_test = test_num_full.shape[0]

    if not FORCE_REBUILD_BASE:
        loaded = try_load_prediction_checkpoint(
            NN_BASE_PRED_CKPT_PATH,
            model_name="nn_base_internal",
            n_train=n_train,
            n_test=n_test,
            n_targets=n_targets,
        )
        if loaded is not None:
            return loaded

    cached = maybe_load_cache(cache_name, force_rebuild=FORCE_REBUILD_BASE)
    if cached is not None:
        oof_preds = cached["oof_preds"].astype(np.float32)
        test_preds = cached["test_preds"].astype(np.float32)
        validate_prediction_shapes(oof_preds, test_preds, n_train, n_test, n_targets, "nn_base_cache")
        save_prediction_checkpoint(
            NN_BASE_PRED_CKPT_PATH,
            "nn_base_internal",
            oof_preds,
            test_preds,
            extra_meta={"source": "cache"},
        )
        return oof_preds, test_preds

    # ---------- DAE over raw numerical subset only ----------
    engineered_prefixes = (
        "is_null_", "null_count_", "null_pca_", "freq_", "diff_", "ratio_", "interact_",
        "row_", "main_row_", "extra_row_"
    )
    dae_idx = [i for i, c in enumerate(num_feature_names) if not any(c.startswith(p) for p in engineered_prefixes)]
    if len(dae_idx) == 0:
        raise RuntimeError("DAE input columns were not found after feature engineering")

    print(f"NN best: DAE input = {len(dae_idx)} raw numerical features")
    train_raw = train_num_full[:, dae_idx].astype(np.float32)
    test_raw = test_num_full[:, dae_idx].astype(np.float32)
    train_null_mask = np.isnan(train_raw).astype(np.uint8)
    test_null_mask = np.isnan(test_raw).astype(np.uint8)
    all_null_mask = np.vstack([train_null_mask, test_null_mask])

    all_num_data = np.vstack([
        np.nan_to_num(train_raw, nan=0.0).astype(np.float32),
        np.nan_to_num(test_raw, nan=0.0).astype(np.float32),
    ])

    dae_mean = all_num_data.mean(axis=0)
    dae_std = all_num_data.std(axis=0)
    dae_std[dae_std < 1e-8] = 1.0
    all_num_data = ((all_num_data - dae_mean) / dae_std).astype(np.float32)

    dae_model = train_dae_on_numpy(all_num_data, all_null_mask, device, fold_dir)
    all_embeddings = extract_dae_embeddings(dae_model, all_num_data, device)

    train_embeddings = all_embeddings[:n_train]
    test_embeddings = all_embeddings[n_train:]
    print(f"NN best: DAE embeddings = {train_embeddings.shape[1]} dims")

    del train_raw, test_raw, train_null_mask, test_null_mask, all_null_mask, all_num_data, dae_model, all_embeddings
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    if device.type == "mps":
        torch.mps.empty_cache()

    train_num_aug = np.hstack([train_num_full.astype(np.float32), train_embeddings.astype(np.float32)]).astype(np.float32)
    test_num_aug = np.hstack([test_num_full.astype(np.float32), test_embeddings.astype(np.float32)]).astype(np.float32)

    # ---------- resume per-fold if available ----------
    oof_preds = np.zeros((n_train, n_targets), dtype=np.float32)
    test_preds_sum = np.zeros((n_test, n_targets), dtype=np.float32)
    fold_aucs = []

    for fi in range(len(folds)):
        ckpt_path = fold_dir / f"fold_{fi}.npz"
        if ckpt_path.exists():
            ckpt = np.load(ckpt_path)
            oof_preds[ckpt["val_idx"]] = ckpt["val_preds"].astype(np.float32)
            test_preds_sum += ckpt["test_preds"].astype(np.float32)
            fold_aucs.append(float(ckpt["fold_auc"]))
            print(f"  Loaded NN fold {fi+1} (AUC={float(ckpt['fold_auc']):.4f})")
        else:
            break

    resume_fold = len(fold_aucs)

    for fold_idx, (tr_idx, val_idx) in enumerate(folds):
        if fold_idx < resume_fold:
            continue

        print(f"\n  ── NN Fold {fold_idx+1}/{len(folds)} ──", flush=True)

        tr_cat = train_cat_full[tr_idx].astype(np.int64)
        val_cat = train_cat_full[val_idx].astype(np.int64)
        te_cat = test_cat_full.astype(np.int64)

        tr_num = train_num_aug[tr_idx]
        val_num = train_num_aug[val_idx]
        te_num = test_num_aug.copy()

        tr_num_q, val_num_q, te_num_q = quantile_normalize_np(tr_num, val_num, te_num)

        tr_y = y_train[tr_idx].astype(np.float32)
        val_y = y_train[val_idx].astype(np.float32)

        val_preds, test_preds, fold_auc = train_one_fold_best_nn(
            fold_idx=fold_idx,
            tr_cat=tr_cat,
            tr_num=tr_num_q,
            tr_y=tr_y,
            val_cat=val_cat,
            val_num=val_num_q,
            val_y=val_y,
            test_cat=te_cat,
            test_num=te_num_q,
            cat_cardinalities=cat_cardinalities,
            target_cols=target_cols,
            device=device,
        )

        oof_preds[val_idx] = val_preds
        test_preds_sum += test_preds
        fold_aucs.append(float(fold_auc))

        np.savez_compressed(
            fold_dir / f"fold_{fold_idx}.npz",
            val_idx=val_idx,
            val_preds=val_preds.astype(np.float32),
            test_preds=test_preds.astype(np.float32),
            fold_auc=np.float32(fold_auc),
        )

        del tr_cat, val_cat, te_cat, tr_num, val_num, te_num, tr_num_q, val_num_q, te_num_q, tr_y, val_y
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
        if device.type == "mps":
            torch.mps.empty_cache()

    test_preds_avg = (test_preds_sum / len(folds)).astype(np.float32)
    final_auc = macro_auc(y_train, oof_preds)
    print("NN fold AUCs:", [round(x, 4) for x in fold_aucs])
    print(f"NN final OOF Macro AUC: {final_auc:.6f}")

    np.savez_compressed(CACHE_DIR / cache_name, oof_preds=oof_preds, test_preds=test_preds_avg)
    save_prediction_checkpoint(
        NN_BASE_PRED_CKPT_PATH,
        "nn_base_internal",
        oof_preds,
        test_preds_avg,
        extra_meta={"fold_aucs": [float(x) for x in fold_aucs], "final_auc": float(final_auc)},
    )
    return oof_preds.astype(np.float32), test_preds_avg.astype(np.float32)


In [7]:

# ============================================================
# 6) BASE MODELS: XGBOOST / CATBOOST  (GPU) + score helpers
# ============================================================
def fit_xgb_base(X_train, y_train, X_test, folds, cache_name="base_xgboost.npz"):
    n_train, n_targets = y_train.shape
    n_test = X_test.shape[0]

    if not FORCE_REBUILD_BASE:
        loaded = try_load_prediction_checkpoint(
            XGB_BASE_PRED_CKPT_PATH,
            model_name="xgb_base_internal",
            n_train=n_train,
            n_test=n_test,
            n_targets=n_targets,
        )
        if loaded is not None:
            return loaded

    cached = maybe_load_cache(cache_name, force_rebuild=FORCE_REBUILD_BASE)
    if cached is not None:
        oof_pred = cached["oof_preds"].astype(np.float32)
        test_pred = cached["test_preds"].astype(np.float32)
        validate_prediction_shapes(oof_pred, test_pred, n_train, n_test, n_targets, "xgb_base_cache")
        save_prediction_checkpoint(
            XGB_BASE_PRED_CKPT_PATH,
            "xgb_base_internal",
            oof_pred,
            test_pred,
            extra_meta={"source": "cache"},
        )
        return oof_pred, test_pred

    import xgboost as xgb
    oof_pred = np.zeros((n_train, n_targets), dtype=np.float32)
    test_pred = np.zeros((n_test, n_targets), dtype=np.float32)

    params = dict(
        objective="binary:logistic",
        eval_metric="auc",
        learning_rate=XGB_LR,
        max_depth=6,
        min_child_weight=20,
        subsample=0.85,
        colsample_bytree=0.75,
        reg_alpha=1.0,
        reg_lambda=6.0,
        n_estimators=XGB_ESTIMATORS,
        max_bin=XGB_MAX_BIN,
        tree_method="hist",
        device="cuda",
        random_state=SEED,
        n_jobs=CPU_COUNT,
        verbosity=0,
        early_stopping_rounds=XGB_EARLY_STOP,
    )

    def worker(j):
        yj = y_train[:, j]
        oof_col = np.zeros(n_train, dtype=np.float32)
        test_col = np.zeros(n_test, dtype=np.float32)

        for tr_idx, va_idx in folds:
            model = xgb.XGBClassifier(**params)
            model.fit(
                X_train[tr_idx],
                yj[tr_idx],
                eval_set=[(X_train[va_idx], yj[va_idx])],
                verbose=False,
            )
            oof_col[va_idx] = model.predict_proba(X_train[va_idx])[:, 1].astype(np.float32)
            test_col += (model.predict_proba(X_test)[:, 1] / len(folds)).astype(np.float32)

        return j, oof_col, test_col

    # GPU models train sequentially to avoid CUDA contention.
    results = parallel_map_targets(worker, n_targets=n_targets, max_workers=1, desc="XGB base")
    for j, oof_col, test_col in results:
        oof_pred[:, j] = oof_col
        test_pred[:, j] = test_col

    np.savez_compressed(CACHE_DIR / cache_name, oof_preds=oof_pred, test_preds=test_pred)
    save_prediction_checkpoint(
        XGB_BASE_PRED_CKPT_PATH,
        "xgb_base_internal",
        oof_pred,
        test_pred,
    )
    return oof_pred.astype(np.float32), test_pred.astype(np.float32)


def fit_catboost_base(train_df_cb, y_train, test_df_cb, folds, cat_cols, cache_name="base_catboost.npz"):
    n_train, n_targets = y_train.shape
    n_test = len(test_df_cb)

    if not FORCE_REBUILD_BASE:
        loaded = try_load_prediction_checkpoint(
            CAT_BASE_PRED_CKPT_PATH,
            model_name="cat_base_internal",
            n_train=n_train,
            n_test=n_test,
            n_targets=n_targets,
        )
        if loaded is not None:
            return loaded

    cached = maybe_load_cache(cache_name, force_rebuild=FORCE_REBUILD_BASE)
    if cached is not None:
        oof_pred = cached["oof_preds"].astype(np.float32)
        test_pred = cached["test_preds"].astype(np.float32)
        validate_prediction_shapes(oof_pred, test_pred, n_train, n_test, n_targets, "cat_base_cache")
        save_prediction_checkpoint(
            CAT_BASE_PRED_CKPT_PATH,
            "cat_base_internal",
            oof_pred,
            test_pred,
            extra_meta={"source": "cache"},
        )
        return oof_pred, test_pred

    from catboost import CatBoostClassifier, Pool

    cat_feature_idx = [train_df_cb.columns.get_loc(c) for c in cat_cols]

    # 1) режем фолды ОДИН раз
    fold_parts = []
    for tr_idx, va_idx in folds:
        fold_parts.append((
            train_df_cb.iloc[tr_idx].copy(),
            train_df_cb.iloc[va_idx].copy(),
            tr_idx,
            va_idx,
        ))

    # 2) test_pool создаём ОДИН раз
    test_pool = Pool(test_df_cb, cat_features=cat_feature_idx)

    params = dict(
        loss_function="Logloss",
        eval_metric="Logloss",          # было AUC -> это главный speedup
        custom_metric=["AUC"],
        learning_rate=CAT_LR,
        iterations=CAT_ITERATIONS,
        depth=6,
        l2_leaf_reg=8.0,
        bootstrap_type="Bernoulli",
        subsample=0.8,
        border_count=32,
        one_hot_max_size=32,
        max_ctr_complexity=1,
        leaf_estimation_iterations=1,
        random_seed=SEED,
        allow_writing_files=False,
        od_type="Iter",
        od_wait=CAT_EARLY_STOP,
        task_type="GPU",
        devices="0",
        metric_period=50,
        verbose=False,
    )

    partial_path = CACHE_DIR / "base_catboost_partial.npz"
    if partial_path.exists():
        saved = np.load(partial_path, allow_pickle=True)
        oof_pred = saved["oof_preds"].astype(np.float32)
        test_pred = saved["test_preds"].astype(np.float32)
        done_targets = int(saved["done_targets"])
        print(f"Resume CatBoost from target {done_targets}/{n_targets}")
    else:
        oof_pred = np.zeros((n_train, n_targets), dtype=np.float32)
        test_pred = np.zeros((n_test, n_targets), dtype=np.float32)
        done_targets = 0

    for j in range(done_targets, n_targets):
        yj = y_train[:, j].astype(np.int32)
        oof_col = np.zeros(n_train, dtype=np.float32)
        test_col = np.zeros(n_test, dtype=np.float32)

        for X_tr_fold, X_va_fold, tr_idx, va_idx in fold_parts:
            train_pool = Pool(X_tr_fold, label=yj[tr_idx], cat_features=cat_feature_idx)
            valid_pool = Pool(X_va_fold, label=yj[va_idx], cat_features=cat_feature_idx)

            model = CatBoostClassifier(**params)
            model.fit(train_pool, eval_set=valid_pool, use_best_model=True, verbose=False)

            oof_col[va_idx] = model.predict_proba(valid_pool)[:, 1].astype(np.float32)
            test_col += (model.predict_proba(test_pool)[:, 1] / len(folds)).astype(np.float32)

        oof_pred[:, j] = oof_col
        test_pred[:, j] = test_col

        np.savez_compressed(
            partial_path,
            oof_preds=oof_pred,
            test_preds=test_pred,
            done_targets=j + 1,
        )
        print(f"CAT base: {j+1:02d}/{n_targets}")

    np.savez_compressed(CACHE_DIR / cache_name, oof_preds=oof_pred, test_preds=test_pred)
    save_prediction_checkpoint(
        CAT_BASE_PRED_CKPT_PATH,
        "cat_base_internal",
        oof_pred,
        test_pred,
    )
    return oof_pred.astype(np.float32), test_pred.astype(np.float32)


def ensure_prob_like(a):
    a = np.asarray(a, dtype=np.float32)
    mn = float(np.nanmin(a))
    mx = float(np.nanmax(a))
    if mn < -1e-5 or mx > 1.00001:
        a = 1.0 / (1.0 + np.exp(-a))
    return np.clip(a, 1e-6, 1 - 1e-6).astype(np.float32)

def flatten_binary_scores(pred, n_rows_expected=None):
    pred = np.asarray(pred)
    if pred.ndim == 1:
        out = pred.astype(np.float32)
    elif pred.ndim == 2:
        if pred.shape[1] == 1:
            out = pred[:, 0].astype(np.float32)
        elif pred.shape[1] == 2:
            out = pred[:, 1].astype(np.float32)
        elif pred.shape[0] == 1:
            out = pred.reshape(-1).astype(np.float32)
        else:
            out = pred.reshape(pred.shape[0], -1)[:, 0].astype(np.float32)
    else:
        out = pred.reshape(-1).astype(np.float32)

    if n_rows_expected is not None and out.shape[0] != n_rows_expected:
        raise ValueError(f"Unexpected flattened prediction length {out.shape[0]} != {n_rows_expected}")
    return ensure_prob_like(out)


In [8]:
# ============================================================
# 7) BUILD BASE MODEL PREDICTIONS:
#    load LGBM/PYBOOST checkpoints
#    train NN/XGBoost/CatBoost from scratch
# ============================================================
print_internal_checkpoint_status(n_train=n_train, n_test=n_test, n_targets=n_targets)

oof_dict = {}
test_dict = {}

if USE_LGBM_CHECKPOINT:
    lgbm_oof, lgbm_test = load_checkpoint_oof_test(LGBM_CKPT_PATH, model_name="lgbm_ckpt")
    validate_prediction_shapes(lgbm_oof, lgbm_test, n_train=n_train, n_test=n_test, n_targets=n_targets, model_name="lgbm_ckpt")
    oof_dict["lgbm"] = lgbm_oof
    test_dict["lgbm"] = lgbm_test
else:
    raise RuntimeError("This notebook expects USE_LGBM_CHECKPOINT=True")

if USE_PYBOOST_CHECKPOINT:
    pyboost_oof, pyboost_test = load_checkpoint_oof_test(PYBOOST_CKPT_PATH, model_name="pyboost_ckpt")
    validate_prediction_shapes(pyboost_oof, pyboost_test, n_train=n_train, n_test=n_test, n_targets=n_targets, model_name="pyboost_ckpt")
    oof_dict["pyboost"] = pyboost_oof
    test_dict["pyboost"] = pyboost_test
else:
    raise RuntimeError("This notebook expects USE_PYBOOST_CHECKPOINT=True")

if USE_NN_CHECKPOINT:
    nn_oof, nn_test = load_checkpoint_oof_test(NN_BASE_PRED_CKPT_PATH, model_name="nn_base")
    validate_prediction_shapes(nn_oof, nn_test, n_train=n_train, n_test=n_test, n_targets=n_targets, model_name="nn_base")
    oof_dict["nn"] = nn_oof
    test_dict["nn"] = nn_test
elif ENABLE_NN:
    oof_dict["nn"], test_dict["nn"] = fit_nn_base_best(
        train_num_full=train_num_np,
        test_num_full=test_num_np,
        train_cat_full=train_cat_np,
        test_cat_full=test_cat_np,
        y_train=y,
        num_feature_names=num_cols,
        target_cols=target_cols,
        cat_cardinalities=cat_cardinalities,
        folds=base_folds,
        cache_name="base_nn_best.npz",
    )
else:
    raise RuntimeError("Need nn checkpoint or ENABLE_NN=True")


if USE_XGB_CHECKPOINT:
    xgb_oof, xgb_test = load_checkpoint_oof_test(XGB_BASE_PRED_CKPT_PATH, model_name="xgb_base")
    validate_prediction_shapes(xgb_oof, xgb_test, n_train=n_train, n_test=n_test, n_targets=n_targets, model_name="xgb_base")
    oof_dict["xgboost"] = xgb_oof
    test_dict["xgboost"] = xgb_test
elif ENABLE_XGB:
    oof_dict["xgboost"], test_dict["xgboost"] = fit_xgb_base(
        X_train=train_tree_np,
        y_train=y,
        X_test=test_tree_np,
        folds=base_folds,
        cache_name="base_xgboost.npz",
    )
else:
    raise RuntimeError("Need xgboost checkpoint or ENABLE_XGB=True")


if USE_CATBOOST_CHECKPOINT:
    cat_oof, cat_test = load_checkpoint_oof_test(CAT_BASE_PRED_CKPT_PATH, model_name="cat_base")
    validate_prediction_shapes(cat_oof, cat_test, n_train=n_train, n_test=n_test, n_targets=n_targets, model_name="cat_base")
    oof_dict["catboost"] = cat_oof
    test_dict["catboost"] = cat_test
elif ENABLE_CATBOOST:
    oof_dict["catboost"], test_dict["catboost"] = fit_catboost_base(
        train_df_cb=train_catboost_df,
        y_train=y,
        test_df_cb=test_catboost_df,
        folds=base_folds,
        cat_cols=cat_cols,
        cache_name="base_catboost.npz",
    )
else:
    raise RuntimeError("Need catboost checkpoint or ENABLE_CATBOOST=True")

required_models = ["nn", "lgbm", "pyboost", "xgboost", "catboost"]
missing = [m for m in required_models if m not in oof_dict]
if missing:
    raise RuntimeError(f"Missing required base models: {missing}")

base_model_names = required_models

np.savez_compressed(
    CACHE_DIR / "base_predictions_all.npz",
    **{f"oof_{k}": v for k, v in oof_dict.items()},
    **{f"test_{k}": v for k, v in test_dict.items()},
)

base_scores = {name: macro_auc(y, oof_dict[name]) for name in base_model_names}
print_scores_table(base_scores, "Base models OOF")

base_corr = mean_rank_correlation_matrix(oof_dict, base_model_names)
print("\nMean flattened rank-correlation between base models:")
print(base_corr)



Internal base-model checkpoint status:
[nn_base] loaded from /kaggle/input/datasets/dderggft/checks/checks/checks/nn_best_predictions.npz
[nn_base] keys: oof_preds, test_preds
[nn_base] shapes: oof=(750000, 41), test=(250000, 41)
  nn_base    -> OK      | /kaggle/input/datasets/dderggft/checks/checks/checks/nn_best_predictions.npz
[xgb_base] loaded from /kaggle/input/datasets/dderggft/checks/checks/checks/xgboost_predictions.npz
[xgb_base] keys: oof_preds, test_preds
[xgb_base] shapes: oof=(750000, 41), test=(250000, 41)
  xgb_base   -> OK      | /kaggle/input/datasets/dderggft/checks/checks/checks/xgboost_predictions.npz
[cat_base] loaded from /kaggle/input/datasets/dderggft/checks/catboost_predictions.npz
[cat_base] keys: oof_preds, test_preds
[cat_base] shapes: oof=(750000, 41), test=(250000, 41)
  cat_base   -> OK      | /kaggle/input/datasets/dderggft/checks/catboost_predictions.npz
[lgbm_ckpt] loaded from /kaggle/input/datasets/dderggft/checks/checks/checks/lgbm_predictions (2).

In [9]:
# ============================================================
# 8) HONEST BLENDS
# ============================================================
def optimize_targetwise_simplex3_subset(y_true, r0, r1, r2, coarse_grid=COARSE_GRID, fine_grid=FINE_GRID):
    n_rows, n_targets = r0.shape

    def worker(j):
        yj = y_true[:, j]
        best_auc = -1.0
        best_w = (1/3, 1/3, 1/3)

        for w0 in coarse_grid:
            for w1 in coarse_grid:
                if w0 + w1 > 1.0:
                    continue
                w2 = round(1.0 - w0 - w1, 10)
                if w2 < 0:
                    continue
                pred = w0 * r0[:, j] + w1 * r1[:, j] + w2 * r2[:, j]
                auc = safe_auc(yj, pred)
                if auc > best_auc:
                    best_auc = auc
                    best_w = (float(w0), float(w1), float(w2))

        w0b, w1b, _ = best_w
        ref0 = [x for x in fine_grid if abs(x - w0b) <= 0.08]
        ref1 = [x for x in fine_grid if abs(x - w1b) <= 0.08]
        for w0 in ref0:
            for w1 in ref1:
                if w0 + w1 > 1.0:
                    continue
                w2 = round(1.0 - w0 - w1, 10)
                if w2 < 0:
                    continue
                pred = w0 * r0[:, j] + w1 * r1[:, j] + w2 * r2[:, j]
                auc = safe_auc(yj, pred)
                if auc > best_auc:
                    best_auc = auc
                    best_w = (float(w0), float(w1), float(w2))

        return j, np.array(best_w, dtype=np.float32), np.float32(best_auc)

    results = parallel_map_targets(worker, n_targets=n_targets, max_workers=PARALLEL_TARGET_WORKERS, desc="simplex3")
    weights = np.zeros((n_targets, 3), dtype=np.float32)
    aucs = np.zeros(n_targets, dtype=np.float32)
    for j, w, auc in results:
        weights[j] = w
        aucs[j] = auc

    return normalize_rows_if_needed(weights), aucs

def apply_simplex3_weights(r0, r1, r2, weights, row_idx=None):
    a0 = r0 if row_idx is None else r0[row_idx]
    a1 = r1 if row_idx is None else r1[row_idx]
    a2 = r2 if row_idx is None else r2[row_idx]
    out = (
        a0 * weights[:, 0][None, :] +
        a1 * weights[:, 1][None, :] +
        a2 * weights[:, 2][None, :]
    )
    return out.astype(np.float32)

def crossfit_simplex3_rank_blend(y_true, oof_r0, oof_r1, oof_r2, test_r0, test_r1, test_r2, n_folds=5, seed=42, cache_name="blend3_cf.npz"):
    cached = maybe_load_cache(cache_name, force_rebuild=FORCE_REBUILD_FINAL)
    if cached is not None:
        return cached["oof_preds"].astype(np.float32), cached["test_preds"].astype(np.float32), cached["fold_weights"].astype(np.float32)

    n_rows, n_targets = y_true.shape
    n_test = test_r0.shape[0]
    oof_pred = np.zeros((n_rows, n_targets), dtype=np.float32)
    test_pred = np.zeros((n_test, n_targets), dtype=np.float32)
    fold_weights = []

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(n_rows)), 1):
        w_fold, aucs_fold = optimize_targetwise_simplex3_subset(
            y_true=y_true[tr_idx],
            r0=oof_r0[tr_idx],
            r1=oof_r1[tr_idx],
            r2=oof_r2[tr_idx],
        )
        oof_pred[va_idx] = apply_simplex3_weights(oof_r0, oof_r1, oof_r2, w_fold, row_idx=va_idx)
        test_pred += apply_simplex3_weights(test_r0, test_r1, test_r2, w_fold) / n_folds
        fold_weights.append(w_fold)
        print(f"blend3_cf fold {fold}/{n_folds} | mean inner auc: {float(np.mean(aucs_fold)):.6f}")

    fold_weights = np.stack(fold_weights, axis=0).astype(np.float32)
    np.savez_compressed(CACHE_DIR / cache_name, oof_preds=oof_pred, test_preds=test_pred, fold_weights=fold_weights)
    return oof_pred, test_pred, fold_weights

def fit_sparse_rank_blend_subset_weights(y_true, rank_dict, model_names, max_models=4, grid=HOLDOUT_GRID):
    n_rows, n_targets = y_true.shape
    n_models = len(model_names)

    def worker(j):
        yj = y_true[:, j]
        best_single_auc = -1.0
        best_single_name = None

        for name in model_names:
            auc = safe_auc(yj, rank_dict[name][:, j])
            if auc > best_single_auc:
                best_single_auc = auc
                best_single_name = name

        cur_w = {name: 0.0 for name in model_names}
        cur_w[best_single_name] = 1.0
        current_pred = rank_dict[best_single_name][:, j].copy()
        current_auc = best_single_auc
        used = {best_single_name}

        for _ in range(max_models - 1):
            best_step = None
            best_step_auc = current_auc
            for cand in model_names:
                if cand in used:
                    continue
                cand_pred = rank_dict[cand][:, j]
                for alpha in grid:
                    mix = (1.0 - alpha) * current_pred + alpha * cand_pred
                    auc = safe_auc(yj, mix)
                    if auc > best_step_auc + 1e-7:
                        best_step_auc = auc
                        best_step = (cand, alpha, mix.astype(np.float32))

            if best_step is None:
                break

            cand, alpha, mix = best_step
            for name in cur_w:
                cur_w[name] *= (1.0 - alpha)
            cur_w[cand] += alpha

            current_pred = mix
            current_auc = best_step_auc
            used.add(cand)

        w = np.array([cur_w[name] for name in model_names], dtype=np.float32)
        return j, w, np.float32(current_auc)

    results = parallel_map_targets(worker, n_targets=n_targets, max_workers=PARALLEL_TARGET_WORKERS, desc="sparse_subset")
    weights = np.zeros((n_targets, n_models), dtype=np.float32)
    aucs = np.zeros(n_targets, dtype=np.float32)
    for j, w, auc in results:
        weights[j] = w
        aucs[j] = auc

    return normalize_rows_if_needed(weights), aucs

def crossfit_sparse_rank_blend(y_true, oof_models, test_models, model_names, n_folds=5, seed=42, max_models=4, grid=HOLDOUT_GRID, cache_name="rank_sparse_cf.npz"):
    cached = maybe_load_cache(cache_name, force_rebuild=FORCE_REBUILD_FINAL)
    if cached is not None:
        return cached["oof_preds"].astype(np.float32), cached["test_preds"].astype(np.float32), cached["fold_weights"].astype(np.float32)

    n_rows, n_targets = y_true.shape
    n_test = next(iter(test_models.values())).shape[0]

    rank_oof = {name: percentile_ranks(oof_models[name]) for name in model_names}
    rank_test = {name: percentile_ranks(test_models[name]) for name in model_names}

    oof_pred = np.zeros((n_rows, n_targets), dtype=np.float32)
    test_pred = np.zeros((n_test, n_targets), dtype=np.float32)
    fold_weights = []

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)

    for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(n_rows)), 1):
        rank_sub = {name: rank_oof[name][tr_idx] for name in model_names}
        w_fold, aucs_fold = fit_sparse_rank_blend_subset_weights(
            y_true=y_true[tr_idx],
            rank_dict=rank_sub,
            model_names=model_names,
            max_models=max_models,
            grid=grid,
        )
        oof_pred[va_idx] = apply_weight_matrix(rank_oof, model_names, w_fold, row_idx=va_idx)
        test_pred += apply_weight_matrix(rank_test, model_names, w_fold) / n_folds
        fold_weights.append(w_fold)
        print(f"{cache_name} fold {fold}/{n_folds} | mean inner auc: {float(np.mean(aucs_fold)):.6f}")

    fold_weights = np.stack(fold_weights, axis=0).astype(np.float32)
    np.savez_compressed(CACHE_DIR / cache_name, oof_preds=oof_pred, test_preds=test_pred, fold_weights=fold_weights)
    return oof_pred, test_pred, fold_weights

In [10]:
# ============================================================
# 9) BUILD HONEST L1 BLENDS FROM BASE MODELS
# ============================================================
if all(k in oof_dict for k in ["nn", "lgbm", "pyboost"]):
    r_nn = percentile_ranks(oof_dict["nn"])
    r_lgb = percentile_ranks(oof_dict["lgbm"])
    r_pb = percentile_ranks(oof_dict["pyboost"])

    rt_nn = percentile_ranks(test_dict["nn"])
    rt_lgb = percentile_ranks(test_dict["lgbm"])
    rt_pb = percentile_ranks(test_dict["pyboost"])

    blend3_cf_oof, blend3_cf_test, blend3_cf_fold_weights = crossfit_simplex3_rank_blend(
        y_true=y,
        oof_r0=r_nn,
        oof_r1=r_lgb,
        oof_r2=r_pb,
        test_r0=rt_nn,
        test_r1=rt_lgb,
        test_r2=rt_pb,
        n_folds=N_FOLDS_BASE,
        seed=SEED,
        cache_name="blend3_cf.npz",
    )
else:
    top3 = sorted(base_scores.items(), key=lambda x: -x[1])[:3]
    a, b, c = [x[0] for x in top3]

    blend3_cf_oof, blend3_cf_test, blend3_cf_fold_weights = crossfit_simplex3_rank_blend(
        y_true=y,
        oof_r0=percentile_ranks(oof_dict[a]),
        oof_r1=percentile_ranks(oof_dict[b]),
        oof_r2=percentile_ranks(oof_dict[c]),
        test_r0=percentile_ranks(test_dict[a]),
        test_r1=percentile_ranks(test_dict[b]),
        test_r2=percentile_ranks(test_dict[c]),
        n_folds=N_FOLDS_BASE,
        seed=SEED,
        cache_name="blend3_cf.npz",
    )

oof_dict["blend3_cf"] = blend3_cf_oof
test_dict["blend3_cf"] = blend3_cf_test

l1_sparse_base_names = list(base_model_names)
rank_sparse_cf_oof, rank_sparse_cf_test, rank_sparse_cf_fold_weights = crossfit_sparse_rank_blend(
    y_true=y,
    oof_models=oof_dict,
    test_models=test_dict,
    model_names=l1_sparse_base_names,
    n_folds=N_FOLDS_BASE,
    seed=SEED,
    max_models=min(MAX_L1_MODELS_PER_TARGET, len(l1_sparse_base_names)),
    grid=HOLDOUT_GRID,
    cache_name="rank_sparse_cf.npz",
)
oof_dict["rank_sparse_cf"] = rank_sparse_cf_oof
test_dict["rank_sparse_cf"] = rank_sparse_cf_test

scores_after_blends = {name: macro_auc(y, pred) for name, pred in oof_dict.items()}
print_scores_table(scores_after_blends, "OOF after honest base blends")

simplex3: 05/41
simplex3: 10/41
simplex3: 15/41
simplex3: 20/41
simplex3: 25/41
simplex3: 30/41
simplex3: 35/41
simplex3: 40/41
simplex3: 41/41
blend3_cf fold 1/5 | mean inner auc: 0.847403
simplex3: 05/41
simplex3: 10/41
simplex3: 15/41
simplex3: 20/41
simplex3: 25/41
simplex3: 30/41
simplex3: 35/41
simplex3: 40/41
simplex3: 41/41
blend3_cf fold 2/5 | mean inner auc: 0.847201
simplex3: 05/41
simplex3: 10/41
simplex3: 15/41
simplex3: 20/41
simplex3: 25/41
simplex3: 30/41
simplex3: 35/41
simplex3: 40/41
simplex3: 41/41
blend3_cf fold 3/5 | mean inner auc: 0.847486
simplex3: 05/41
simplex3: 10/41
simplex3: 15/41
simplex3: 20/41
simplex3: 25/41
simplex3: 30/41
simplex3: 35/41
simplex3: 40/41
simplex3: 41/41
blend3_cf fold 4/5 | mean inner auc: 0.846082
simplex3: 05/41
simplex3: 10/41
simplex3: 15/41
simplex3: 20/41
simplex3: 25/41
simplex3: 30/41
simplex3: 35/41
simplex3: 40/41
simplex3: 41/41
blend3_cf fold 5/5 | mean inner auc: 0.846474
sparse_subset: 05/41
sparse_subset: 10/41
sparse_s

In [11]:

# ============================================================
# 10) META LEARNERS
# ============================================================
def fit_ridge_meta_exact(
    X_train,
    y_train,
    X_test,
    n_folds=5,
    seed=42,
    cache_name="ridge_meta_exact.npz",
    alphas=(0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0),
    fit_intercept=True,
    standardize=True,
    math_precision="float64",
    prefer_gpu=True,
):
    cached = maybe_load_cache(cache_name, force_rebuild=FORCE_REBUILD_META)
    if cached is not None:
        return cached["oof_preds"].astype(np.float32), cached["test_preds"].astype(np.float32)

    X_train = np.asarray(X_train, dtype=np.float32, order="C")
    y_train = np.asarray(y_train, dtype=np.float32, order="C")
    X_test = np.asarray(X_test, dtype=np.float32, order="C")

    n_train, n_features = X_train.shape
    n_test = X_test.shape[0]
    n_targets = y_train.shape[1]
    n_alphas = len(alphas)

    backend_name = "numpy"
    xp = np
    to_numpy = np.asarray

    if prefer_gpu:
        try:
            import cupy as cp
            xp = cp
            to_numpy = cp.asnumpy
            backend_name = "cupy"
        except Exception:
            pass

    backend_dtype = xp.float64 if math_precision == "float64" else xp.float32

    oof_by_alpha = np.zeros((n_alphas, n_train, n_targets), dtype=np.float32)
    test_by_alpha = np.zeros((n_alphas, n_test, n_targets), dtype=np.float32)

    X_test_backend = xp.asarray(X_test, dtype=backend_dtype)
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)

    for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(n_train)), 1):
        X_tr = xp.asarray(X_train[tr_idx], dtype=backend_dtype)
        Y_tr = xp.asarray(y_train[tr_idx], dtype=backend_dtype)
        X_va = xp.asarray(X_train[va_idx], dtype=backend_dtype)

        if fit_intercept:
            x_mean = X_tr.mean(axis=0, keepdims=True)
            y_mean = Y_tr.mean(axis=0, keepdims=True)
        else:
            x_mean = xp.zeros((1, n_features), dtype=backend_dtype)
            y_mean = xp.zeros((1, n_targets), dtype=backend_dtype)

        X_tr_c = X_tr - x_mean
        X_va_c = X_va - x_mean
        X_te_c = X_test_backend - x_mean
        Y_tr_c = Y_tr - y_mean

        if standardize:
            x_std = X_tr_c.std(axis=0, keepdims=True)
            x_std = xp.where(x_std < 1e-8, 1.0, x_std)
            X_tr_c = X_tr_c / x_std
            X_va_c = X_va_c / x_std
            X_te_c = X_te_c / x_std

        XtX = X_tr_c.T @ X_tr_c
        XtY = X_tr_c.T @ Y_tr_c
        evals, evecs = xp.linalg.eigh(XtX)
        Qt_XtY = evecs.T @ XtY

        for a_idx, alpha in enumerate(alphas):
            denom = evals[:, None] + backend_dtype(alpha)
            coef = evecs @ (Qt_XtY / denom)
            va_pred = X_va_c @ coef + y_mean
            te_pred = X_te_c @ coef + y_mean

            oof_by_alpha[a_idx, va_idx, :] = to_numpy(va_pred).astype(np.float32)
            test_by_alpha[a_idx] += (to_numpy(te_pred).astype(np.float32) / n_folds)

        if backend_name == "cupy":
            try:
                import cupy as cp
                cp.get_default_memory_pool().free_all_blocks()
            except Exception:
                pass
        gc.collect()

    def worker(j):
        yj = y_train[:, j]
        aucs_j = np.empty(n_alphas, dtype=np.float32)
        for a_idx in range(n_alphas):
            aucs_j[a_idx] = safe_auc(yj, oof_by_alpha[a_idx, :, j])
        best_idx = int(np.argmax(aucs_j))
        return j, oof_by_alpha[best_idx, :, j], test_by_alpha[best_idx, :, j], np.float32(alphas[best_idx]), np.float32(aucs_j[best_idx])

    results = parallel_map_targets(worker, n_targets=n_targets, max_workers=PARALLEL_TARGET_WORKERS, desc="ridge alpha select")

    best_oof = np.zeros((n_train, n_targets), dtype=np.float32)
    best_test = np.zeros((n_test, n_targets), dtype=np.float32)
    best_alphas = np.zeros(n_targets, dtype=np.float32)
    best_aucs = np.zeros(n_targets, dtype=np.float32)

    for j, oof_col, test_col, alpha, auc in results:
        best_oof[:, j] = oof_col
        best_test[:, j] = test_col
        best_alphas[j] = alpha
        best_aucs[j] = auc

    np.savez_compressed(
        CACHE_DIR / cache_name,
        oof_preds=best_oof,
        test_preds=best_test,
        best_alphas=best_alphas,
        best_aucs=best_aucs,
    )
    return best_oof.astype(np.float32), best_test.astype(np.float32)


def fit_lgb_meta(X_train, y_train, X_test, n_folds=5, seed=42, cache_name="lgb_meta.npz"):
    cached = maybe_load_cache(cache_name, force_rebuild=FORCE_REBUILD_META)
    if cached is not None:
        return cached["oof_preds"].astype(np.float32), cached["test_preds"].astype(np.float32)

    import lightgbm as lgb

    X_train = np.asarray(X_train, dtype=np.float32, order="C")
    y_train = np.asarray(y_train, dtype=np.float32, order="C")
    X_test = np.asarray(X_test, dtype=np.float32, order="C")

    n_train, n_targets = y_train.shape
    n_test = X_test.shape[0]
    oof_pred = np.zeros((n_train, n_targets), dtype=np.float32)
    test_pred = np.zeros((n_test, n_targets), dtype=np.float32)

    params = dict(
        objective="binary",
        metric="auc",
        learning_rate=0.03,
        n_estimators=1200,
        num_leaves=31,
        min_child_samples=400,
        subsample=0.9,
        colsample_bytree=0.8,
        reg_alpha=1.0,
        reg_lambda=4.0,
        random_state=seed,
        n_jobs=CPU_COUNT,
        device_type="cpu",
        verbose=-1,
    )

    def fit_one_target(j):
        yj = y_train[:, j]
        kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
        oof_col = np.zeros(n_train, dtype=np.float32)
        test_col = np.zeros(n_test, dtype=np.float32)

        for tr_idx, va_idx in kf.split(X_train):
            model = lgb.LGBMClassifier(**params)
            model.fit(
                X_train[tr_idx],
                yj[tr_idx],
                eval_set=[(X_train[va_idx], yj[va_idx])],
                callbacks=[lgb.early_stopping(stopping_rounds=80, verbose=False)],
            )
            oof_col[va_idx] = model.predict_proba(X_train[va_idx])[:, 1].astype(np.float32)
            test_col += (model.predict_proba(X_test)[:, 1] / n_folds).astype(np.float32)

        return j, oof_col, test_col

    # LGBM on CPU with all cores -> one target at a time to avoid oversubscription.
    results = parallel_map_targets(fit_one_target, n_targets=n_targets, max_workers=1, desc=f"{cache_name} fit")
    for j, oof_col, test_col in results:
        oof_pred[:, j] = oof_col
        test_pred[:, j] = test_col

    np.savez_compressed(CACHE_DIR / cache_name, oof_preds=oof_pred, test_preds=test_pred)
    return oof_pred.astype(np.float32), test_pred.astype(np.float32)


def fit_xgb_meta(X_train, y_train, X_test, n_folds=5, seed=42, cache_name="xgb_meta.npz"):
    cached = maybe_load_cache(cache_name, force_rebuild=FORCE_REBUILD_META)
    if cached is not None:
        return cached["oof_preds"].astype(np.float32), cached["test_preds"].astype(np.float32)

    import xgboost as xgb

    X_train = np.asarray(X_train, dtype=np.float32, order="C")
    y_train = np.asarray(y_train, dtype=np.float32, order="C")
    X_test = np.asarray(X_test, dtype=np.float32, order="C")

    n_train, n_targets = y_train.shape
    n_test = X_test.shape[0]
    oof_pred = np.zeros((n_train, n_targets), dtype=np.float32)
    test_pred = np.zeros((n_test, n_targets), dtype=np.float32)

    params = dict(
        objective="binary:logistic",
        eval_metric="auc",
        learning_rate=0.03,
        max_depth=5,
        min_child_weight=20,
        subsample=0.85,
        colsample_bytree=0.75,
        reg_alpha=1.0,
        reg_lambda=4.0,
        n_estimators=1600,
        max_bin=256,
        tree_method="hist",
        device="cuda",
        random_state=seed,
        n_jobs=CPU_COUNT,
        verbosity=0,
        early_stopping_rounds=80,
    )

    def fit_one_target(j):
        yj = y_train[:, j]
        kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
        oof_col = np.zeros(n_train, dtype=np.float32)
        test_col = np.zeros(n_test, dtype=np.float32)

        for tr_idx, va_idx in kf.split(X_train):
            model = xgb.XGBClassifier(**params)
            model.fit(X_train[tr_idx], yj[tr_idx], eval_set=[(X_train[va_idx], yj[va_idx])], verbose=False)
            oof_col[va_idx] = model.predict_proba(X_train[va_idx])[:, 1].astype(np.float32)
            test_col += (model.predict_proba(X_test)[:, 1] / n_folds).astype(np.float32)

        return j, oof_col, test_col

    results = parallel_map_targets(fit_one_target, n_targets=n_targets, max_workers=1, desc=f"{cache_name} fit")
    for j, oof_col, test_col in results:
        oof_pred[:, j] = oof_col
        test_pred[:, j] = test_col

    np.savez_compressed(CACHE_DIR / cache_name, oof_preds=oof_pred, test_preds=test_pred)
    return oof_pred.astype(np.float32), test_pred.astype(np.float32)


def fit_cat_meta(X_train, y_train, X_test, n_folds=5, seed=42, cache_name="cat_meta.npz"):
    cached = maybe_load_cache(cache_name, force_rebuild=FORCE_REBUILD_META)
    if cached is not None:
        return cached["oof_preds"].astype(np.float32), cached["test_preds"].astype(np.float32)

    from catboost import CatBoostClassifier

    X_train = np.asarray(X_train, dtype=np.float32, order="C")
    y_train = np.asarray(y_train, dtype=np.float32, order="C")
    X_test = np.asarray(X_test, dtype=np.float32, order="C")

    n_train, n_targets = y_train.shape
    n_test = X_test.shape[0]
    oof_pred = np.zeros((n_train, n_targets), dtype=np.float32)
    test_pred = np.zeros((n_test, n_targets), dtype=np.float32)

    params = dict(
        loss_function="Logloss",
        eval_metric="AUC",
        learning_rate=0.03,
        iterations=1800,
        depth=6,
        l2_leaf_reg=8.0,
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
        od_type="Iter",
        od_wait=80,
        task_type="GPU",
        devices="0",
        thread_count=CPU_COUNT,
    )

    def fit_one_target(j):
        yj = y_train[:, j].astype(np.int32)
        kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
        oof_col = np.zeros(n_train, dtype=np.float32)
        test_col = np.zeros(n_test, dtype=np.float32)

        for tr_idx, va_idx in kf.split(X_train):
            model = CatBoostClassifier(**params)
            model.fit(X_train[tr_idx], yj[tr_idx], eval_set=(X_train[va_idx], yj[va_idx]), use_best_model=True, verbose=False)
            oof_col[va_idx] = model.predict_proba(X_train[va_idx])[:, 1].astype(np.float32)
            test_col += (model.predict_proba(X_test)[:, 1] / n_folds).astype(np.float32)

        return j, oof_col, test_col

    results = parallel_map_targets(fit_one_target, n_targets=n_targets, max_workers=1, desc=f"{cache_name} fit")
    for j, oof_col, test_col in results:
        oof_pred[:, j] = oof_col
        test_pred[:, j] = test_col

    np.savez_compressed(CACHE_DIR / cache_name, oof_preds=oof_pred, test_preds=test_pred)
    return oof_pred.astype(np.float32), test_pred.astype(np.float32)


def fit_pyboost_meta(X_train, y_train, X_test, n_folds=5, seed=42, cache_name="pyboost_meta.npz"):
    cached = maybe_load_cache(cache_name, force_rebuild=FORCE_REBUILD_META)
    if cached is not None:
        return cached["oof_preds"].astype(np.float32), cached["test_preds"].astype(np.float32)

    if not ENABLE_PYBOOST_META:
        raise RuntimeError("ENABLE_PYBOOST_META=False")
    if not PYBOOST_META_READY:
        raise RuntimeError(f"PyBoost meta requested but environment is not ready: {PYBOOST_META_ERROR}")

    from py_boost import GradientBoosting

    X_train = np.asarray(X_train, dtype=np.float32, order="C")
    y_train = np.asarray(y_train, dtype=np.float32, order="C")
    X_test = np.asarray(X_test, dtype=np.float32, order="C")

    n_train, n_targets = y_train.shape
    n_test = X_test.shape[0]
    oof_pred = np.zeros((n_train, n_targets), dtype=np.float32)
    test_pred = np.zeros((n_test, n_targets), dtype=np.float32)

    def fit_one_target(j):
        yj = y_train[:, j].astype(np.float32)
        kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
        oof_col = np.zeros(n_train, dtype=np.float32)
        test_col = np.zeros(n_test, dtype=np.float32)

        for tr_idx, va_idx in kf.split(X_train):
            model = GradientBoosting(
                "crossentropy",
                ntrees=PYBOOST_META_NTREES,
                lr=PYBOOST_META_LR,
                verbose=PYBOOST_META_VERBOSE,
                es=PYBOOST_META_EARLY_STOP,
                lambda_l2=1.0,
                gd_steps=1,
                subsample=0.85,
                colsample=0.75,
                min_data_in_leaf=20,
                use_hess=True,
                max_bin=PYBOOST_META_MAX_BIN,
                max_depth=PYBOOST_META_MAX_DEPTH,
            )
            model.fit(
                X_train[tr_idx].astype(np.float32),
                yj[tr_idx].astype(np.float32),
                eval_sets=[{"X": X_train[va_idx].astype(np.float32), "y": yj[va_idx].astype(np.float32)}],
            )

            va_pred = flatten_binary_scores(model.predict(X_train[va_idx].astype(np.float32)), len(va_idx))
            te_pred = flatten_binary_scores(model.predict(X_test.astype(np.float32)), n_test)

            oof_col[va_idx] = va_pred
            test_col += te_pred / n_folds

        return j, oof_col.astype(np.float32), test_col.astype(np.float32)

    results = parallel_map_targets(fit_one_target, n_targets=n_targets, max_workers=1, desc=f"{cache_name} fit")
    for j, oof_col, test_col in results:
        oof_pred[:, j] = oof_col
        test_pred[:, j] = test_col

    np.savez_compressed(CACHE_DIR / cache_name, oof_preds=oof_pred, test_preds=test_pred)
    return oof_pred.astype(np.float32), test_pred.astype(np.float32)


def build_level_feature_matrices(source_names):
    rank_oof = {name: percentile_ranks(oof_dict[name]) for name in source_names}
    rank_test = {name: percentile_ranks(test_dict[name]) for name in source_names}
    X_train = build_meta_features([oof_dict[name] for name in source_names], rank_list=[rank_oof[name] for name in source_names])
    X_test = build_meta_features([test_dict[name] for name in source_names], rank_list=[rank_test[name] for name in source_names])
    return X_train, X_test


def fit_meta_level(level_tag, source_names):
    print(f"\n==== Meta level {level_tag.upper()} ====")
    print("source names:", source_names)
    if len(source_names) < 2:
        raise ValueError(f"Need at least 2 source models for meta level {level_tag}")

    X_train, X_test = build_level_feature_matrices(source_names)
    print(f"{level_tag.upper()} X_train:", X_train.shape)
    print(f"{level_tag.upper()} X_test :", X_test.shape)

    created_names = []

    ridge_name = f"ridge_meta_{level_tag}"
    ridge_oof, ridge_test = fit_ridge_meta_exact(
        X_train=X_train,
        y_train=y,
        X_test=X_test,
        n_folds=N_FOLDS_META,
        seed=SEED,
        cache_name=f"{ridge_name}.npz",
    )
    oof_dict[ridge_name] = ridge_oof
    test_dict[ridge_name] = ridge_test
    created_names.append(ridge_name)

    if ENABLE_LGB_META:
        lgb_name = f"lgb_meta_{level_tag}"
        lgb_oof, lgb_test = fit_lgb_meta(X_train, y, X_test, n_folds=N_FOLDS_META, seed=SEED, cache_name=f"{lgb_name}.npz")
        oof_dict[lgb_name] = lgb_oof
        test_dict[lgb_name] = lgb_test
        created_names.append(lgb_name)

    if ENABLE_XGB_META:
        xgb_name = f"xgb_meta_{level_tag}"
        xgb_oof, xgb_test = fit_xgb_meta(X_train, y, X_test, n_folds=N_FOLDS_META, seed=SEED, cache_name=f"{xgb_name}.npz")
        oof_dict[xgb_name] = xgb_oof
        test_dict[xgb_name] = xgb_test
        created_names.append(xgb_name)

    if ENABLE_CAT_META:
        cat_name = f"cat_meta_{level_tag}"
        cat_oof, cat_test = fit_cat_meta(X_train, y, X_test, n_folds=N_FOLDS_META, seed=SEED, cache_name=f"{cat_name}.npz")
        oof_dict[cat_name] = cat_oof
        test_dict[cat_name] = cat_test
        created_names.append(cat_name)

    if ENABLE_PYBOOST_META and PYBOOST_META_READY:
        pyb_name = f"pyboost_meta_{level_tag}"
        pyb_oof, pyb_test = fit_pyboost_meta(X_train, y, X_test, n_folds=N_FOLDS_META, seed=SEED, cache_name=f"{pyb_name}.npz")
        oof_dict[pyb_name] = pyb_oof
        test_dict[pyb_name] = pyb_test
        created_names.append(pyb_name)

    level_scores = {name: macro_auc(y, oof_dict[name]) for name in created_names}
    print_scores_table(level_scores, f"OOF scores of meta models at level {level_tag.upper()}")
    return created_names


In [ ]:
preload_available_cached_models(oof_dict, test_dict)

In [ ]:
# ============================================================
# 11) L1 META ON 5 BASE MODELS + HONEST BLENDS
# ============================================================
l1_meta_source_names = [
    name for name in [
        "nn",
        "lgbm",
        "pyboost",
        "xgboost",
        "catboost",
        "blend3_cf",
        "rank_sparse_cf",
    ]
    if name in oof_dict
]

created_l1_names = fit_meta_level("l1", l1_meta_source_names)

scores_after_l1 = {name: macro_auc(y, pred) for name, pred in oof_dict.items()}
print_scores_table(scores_after_l1, "OOF after L1")


==== Meta level L1 ====
source names: ['nn', 'lgbm', 'pyboost', 'xgboost', 'catboost', 'blend3_cf', 'rank_sparse_cf']
L1 X_train: (750000, 1599)
L1 X_test : (250000, 1599)
ridge alpha select: 05/41
ridge alpha select: 10/41
ridge alpha select: 15/41
ridge alpha select: 20/41
ridge alpha select: 25/41
ridge alpha select: 30/41
ridge alpha select: 35/41
ridge alpha select: 40/41
ridge alpha select: 41/41
lgb_meta_l1.npz fit: 05/41
lgb_meta_l1.npz fit: 10/41
lgb_meta_l1.npz fit: 15/41
lgb_meta_l1.npz fit: 20/41
lgb_meta_l1.npz fit: 25/41
lgb_meta_l1.npz fit: 30/41
lgb_meta_l1.npz fit: 35/41
lgb_meta_l1.npz fit: 40/41
lgb_meta_l1.npz fit: 41/41
xgb_meta_l1.npz fit: 05/41
xgb_meta_l1.npz fit: 10/41
xgb_meta_l1.npz fit: 15/41
xgb_meta_l1.npz fit: 20/41
xgb_meta_l1.npz fit: 25/41
xgb_meta_l1.npz fit: 30/41
xgb_meta_l1.npz fit: 35/41
xgb_meta_l1.npz fit: 40/41
xgb_meta_l1.npz fit: 41/41


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric perio

cat_meta_l1.npz fit: 05/41


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric perio

cat_meta_l1.npz fit: 10/41


...

cat_meta_l1.npz fit: 35/41


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric perio

cat_meta_l1.npz fit: 40/41


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


cat_meta_l1.npz fit: 41/41
[17:24:16] Stdout logging level is INFO.
[17:24:16] GDBT train starts. Max iter 2500, early stopping rounds 120
[17:24:21] Iter 0; Sample 0, Crossentropy = 0.6645003959277089; 
[17:25:03] Iter 200; Sample 0, Crossentropy = 0.03536071621868843; 
[17:25:45] Iter 400; Sample 0, Crossentropy = 0.03501831923636303; 
[17:25:50] Early stopping at iter 425, best iter 305, best_score 0.03495994059331191
[17:26:10] Stdout logging level is INFO.
[17:26:10] GDBT train starts. Max iter 2500, early stopping rounds 120
[17:26:10] Iter 0; Sample 0, Crossentropy = 0.6645001574072026; 
[17:26:52] Iter 200; Sample 0, Crossentropy = 0.03559844784128352; 
[17:27:33] Iter 400; Sample 0, Crossentropy = 0.03522501332716115; 
[17:27:42] Early stopping at iter 440, best iter 320, best_score 0.03517225581888016
[17:28:02] Stdout logging level is INFO.
[17:28:02] GDBT train starts. Max iter 2500, early stopping rounds 120
[17:28:02] Iter 0; Sample 0, Crossentropy = 0.6645137530103414; 


In [ ]:

# ============================================================
# 12) L2 META ON L1 PREDICTIONS
# ============================================================
l2_source_names = [
    name for name in [
        "blend3_cf",
        "rank_sparse_cf",
        "ridge_meta_l1",
        "lgb_meta_l1",
        "xgb_meta_l1",
        "cat_meta_l1",
        "pyboost_meta_l1",
    ]
    if name in oof_dict
]

print("L2 source names:", l2_source_names)

if ENABLE_L2 and len(l2_source_names) >= 3:
    created_l2_names = fit_meta_level("l2", l2_source_names)

    rank_sparse_l2_oof, rank_sparse_l2_test, rank_sparse_l2_fold_weights = crossfit_sparse_rank_blend(
        y_true=y,
        oof_models=oof_dict,
        test_models=test_dict,
        model_names=l2_source_names,
        n_folds=N_FOLDS_META,
        seed=SEED,
        max_models=min(MAX_L2_MODELS_PER_TARGET, len(l2_source_names)),
        grid=HOLDOUT_GRID,
        cache_name="rank_sparse_l2_cf.npz",
    )
    oof_dict["rank_sparse_l2_cf"] = rank_sparse_l2_oof
    test_dict["rank_sparse_l2_cf"] = rank_sparse_l2_test
else:
    created_l2_names = []
    print("Skip L2")

scores_after_l2 = {name: macro_auc(y, pred) for name, pred in oof_dict.items()}
print_scores_table(scores_after_l2, "OOF after L2")


In [ ]:

# ============================================================
# 13) L3 META ON L2 PREDICTIONS
# ============================================================
l3_source_names = [
    name for name in [
        "rank_sparse_l2_cf",
        "ridge_meta_l2",
        "lgb_meta_l2",
        "xgb_meta_l2",
        "cat_meta_l2",
        "pyboost_meta_l2",
    ]
    if name in oof_dict
]

print("L3 source names:", l3_source_names)

if ENABLE_L3 and len(l3_source_names) >= 2:
    created_l3_names = fit_meta_level("l3", l3_source_names)

    if len(l3_source_names) >= 3:
        rank_sparse_l3_oof, rank_sparse_l3_test, rank_sparse_l3_fold_weights = crossfit_sparse_rank_blend(
            y_true=y,
            oof_models=oof_dict,
            test_models=test_dict,
            model_names=l3_source_names,
            n_folds=N_FOLDS_META,
            seed=SEED,
            max_models=min(MAX_L3_MODELS_PER_TARGET, len(l3_source_names)),
            grid=HOLDOUT_GRID,
            cache_name="rank_sparse_l3_cf.npz",
        )
        oof_dict["rank_sparse_l3_cf"] = rank_sparse_l3_oof
        test_dict["rank_sparse_l3_cf"] = rank_sparse_l3_test
else:
    created_l3_names = []
    print("Skip L3")

scores_after_l3 = {name: macro_auc(y, pred) for name, pred in oof_dict.items()}
print_scores_table(scores_after_l3, "OOF after L3")


In [ ]:

# ============================================================
# 14) FINAL HONEST SUPERBLEND + FULL-DATA REFIT
# ============================================================
def fit_sparse_rank_blend_full_data(y_true, oof_models, test_models, model_names, max_models=5, grid=HOLDOUT_GRID, cache_name="superblend_fullrefit.npz"):
    cached = maybe_load_cache(cache_name, force_rebuild=FORCE_REBUILD_FINAL)
    if cached is not None:
        return cached["test_preds"].astype(np.float32), cached["weights"].astype(np.float32)

    rank_oof = {name: percentile_ranks(oof_models[name]) for name in model_names}
    rank_test = {name: percentile_ranks(test_models[name]) for name in model_names}

    weights, inner_aucs = fit_sparse_rank_blend_subset_weights(
        y_true=y_true,
        rank_dict=rank_oof,
        model_names=model_names,
        max_models=max_models,
        grid=grid,
    )
    test_pred = apply_weight_matrix(rank_test, model_names, weights)

    np.savez_compressed(
        CACHE_DIR / cache_name,
        test_preds=test_pred.astype(np.float32),
        weights=weights.astype(np.float32),
        inner_aucs=inner_aucs.astype(np.float32),
    )
    return test_pred.astype(np.float32), weights.astype(np.float32)

preferred_final_order = [
    "ridge_meta_l3",
    "lgb_meta_l3",
    "xgb_meta_l3",
    "cat_meta_l3",
    "pyboost_meta_l3",
    "rank_sparse_l3_cf",
    "ridge_meta_l2",
    "lgb_meta_l2",
    "xgb_meta_l2",
    "cat_meta_l2",
    "pyboost_meta_l2",
    "rank_sparse_l2_cf",
    "ridge_meta_l1",
    "lgb_meta_l1",
    "xgb_meta_l1",
    "cat_meta_l1",
    "pyboost_meta_l1",
    "rank_sparse_cf",
    "blend3_cf",
    "nn",
    "lgbm",
    "pyboost",
    "xgboost",
    "catboost",
]

all_scores = {name: macro_auc(y, pred) for name, pred in oof_dict.items()}
final_candidates = [name for name in preferred_final_order if name in oof_dict]
if len(final_candidates) > 16:
    final_candidates = final_candidates[:16]

print("Final candidates:", final_candidates)

superblend_cf_oof, superblend_cf_test, superblend_cf_fold_weights = crossfit_sparse_rank_blend(
    y_true=y,
    oof_models=oof_dict,
    test_models=test_dict,
    model_names=final_candidates,
    n_folds=N_FOLDS_FINAL,
    seed=SEED,
    max_models=min(MAX_FINAL_MODELS_PER_TARGET, len(final_candidates)),
    grid=HOLDOUT_GRID,
    cache_name="superblend_cf.npz",
)
oof_dict["superblend_cf"] = superblend_cf_oof
test_dict["superblend_cf"] = superblend_cf_test

superblend_cf_score = macro_auc(y, superblend_cf_oof)
print("superblend_cf OOF:", f"{superblend_cf_score:.6f}")

superblend_full_test, superblend_full_weights = fit_sparse_rank_blend_full_data(
    y_true=y,
    oof_models=oof_dict,
    test_models=test_dict,
    model_names=final_candidates,
    max_models=min(MAX_FINAL_MODELS_PER_TARGET, len(final_candidates)),
    grid=HOLDOUT_GRID,
    cache_name="superblend_fullrefit.npz",
)
test_dict["superblend_fullrefit"] = superblend_full_test

avg_cf_w = pd.Series(superblend_cf_fold_weights.mean(axis=(0, 1)), index=final_candidates).sort_values(ascending=False)
avg_full_w = pd.Series(superblend_full_weights.mean(axis=0), index=final_candidates).sort_values(ascending=False)

print("\nAverage cross-fit final weights:")
print(avg_cf_w)

print("\nAverage full-data refit weights:")
print(avg_full_w)

weights_dump = {
    "candidate_names": final_candidates,
    "avg_cf_weights": {k: float(v) for k, v in avg_cf_w.to_dict().items()},
    "avg_fullrefit_weights": {k: float(v) for k, v in avg_full_w.to_dict().items()},
}
(CACHE_DIR / "final_superblend_weights.json").write_text(
    json.dumps(weights_dump, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

final_scores = {name: macro_auc(y, pred) for name, pred in oof_dict.items()}
print_scores_table(final_scores, "All honest OOF scores (final)")


In [ ]:

# ============================================================
# 15) SAVE SUBMISSIONS
# ============================================================
save_submission(test_dict["superblend_fullrefit"], "superblend_primary__fullrefit.parquet")
save_submission(test_dict["superblend_cf"], "superblend_secondary__crossfit.parquet")

best_meta_like = None
for cand in [
    "ridge_meta_l3",
    "lgb_meta_l3",
    "xgb_meta_l3",
    "cat_meta_l3",
    "pyboost_meta_l3",
    "rank_sparse_l3_cf",
    "ridge_meta_l2",
    "lgb_meta_l2",
    "xgb_meta_l2",
    "cat_meta_l2",
    "pyboost_meta_l2",
    "rank_sparse_l2_cf",
    "ridge_meta_l1",
    "lgb_meta_l1",
    "xgb_meta_l1",
    "cat_meta_l1",
    "pyboost_meta_l1",
]:
    if cand in test_dict:
        best_meta_like = cand
        break

if best_meta_like is not None:
    meta_heavy = (
        0.85 * percentile_ranks(test_dict["superblend_fullrefit"]) +
        0.15 * percentile_ranks(test_dict[best_meta_like])
    )
    save_submission(meta_heavy, f"superblend_metaheavy__plus_{best_meta_like}.parquet")

best_single_name = max(all_scores.items(), key=lambda x: x[1])[0]
safe_mix = (
    0.90 * percentile_ranks(test_dict["superblend_fullrefit"]) +
    0.10 * percentile_ranks(test_dict[best_single_name])
)
save_submission(safe_mix, f"superblend_safe__plus_{best_single_name}.parquet")
save_submission(test_dict[best_single_name], f"single_best__{best_single_name}.parquet")

for cand in ["blend3_cf", "rank_sparse_cf", "rank_sparse_l2_cf", "rank_sparse_l3_cf"]:
    if cand in test_dict:
        save_submission(test_dict[cand], f"{cand}.parquet")

summary = {
    "base_models": list(base_model_names),
    "scores": {k: float(v) for k, v in sorted(final_scores.items(), key=lambda x: -x[1])},
    "superblend_cf_oof": float(superblend_cf_score),
    "best_single_name": best_single_name,
    "best_single_oof": float(all_scores[best_single_name]),
    "final_candidates": final_candidates,
    "submissions_dir": str(SUB_DIR),
    "cache_dir": str(CACHE_DIR),
    "device_policy": {
        "nn": "cpu",
        "lgbm_meta": "cpu",
        "xgboost_base_meta": "gpu",
        "catboost_base_meta": "gpu",
        "pyboost_meta": "gpu" if ENABLE_PYBOOST_META and PYBOOST_META_READY else "disabled",
        "lgbm_base": "checkpoint",
        "pyboost_base": "checkpoint",
    },
}
(WORK_DIR / "run_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\nDone.")
print("Best single :", best_single_name, f"{all_scores[best_single_name]:.6f}")
print("Primary OOF :", f"{superblend_cf_score:.6f}")
print("Summary     :", WORK_DIR / "run_summary.json")
print("Submissions :", SUB_DIR)


In [ ]:
print("Notebook finished successfully.")
